## 1. Imports and Setup

In [ ]:
import os
import re
import json
from pathlib import Path
from dataclasses import dataclass, replace
from collections import defaultdict
from typing import Any, Dict, List, Optional, Tuple, Iterable

import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
from rasterio.mask import mask
from shapely.geometry import mapping, Polygon
from shapely.affinity import translate
import networkx as nx
from sklearn.neighbors import NearestNeighbors

SKIMAGE_AVAILABLE = True
try:
    from skimage.registration import phase_cross_correlation
    from skimage.transform import resize
except Exception as e:
    SKIMAGE_AVAILABLE = False
    print("⚠️ scikit-image not available. Phase correlation alignment will be skipped.")
    print(str(e))

In [ ]:
@dataclass
class MatchCaseConfig:
    name: str
    base_similarity_weights: Dict[str, float]
    scoring_weights: Dict[str, float]
    similarity_threshold: float
    min_iou: float = 0.0
    min_overlap_prev: float = 0.0
    min_overlap_curr: float = 0.0
    max_centroid_dist: Optional[float] = None
    mutual_best: bool = False
    allow_multiple: bool = False
    max_edges_per_prev: Optional[int] = None
    max_edges_per_curr: Optional[int] = None


class TreeTrackingGraph:
    """
    Tree crown tracker across orthomosaics using a directed graph with ultra-relaxed parameters.
    
    Supports automatic file discovery, image extraction, spatial alignment, and configurable matching.
    """
    
    def __init__(
        self,
        crown_dir: Optional[str] = None,
        ortho_dir: Optional[str] = None,
        output_dir: str = '../../output',
        simplify_tol: float = 1.0,
        max_crowns_preview: int = 200,
        auto_discover: bool = True
    ) -> None:
        self.output_dir = output_dir
        self.simplify_tol = simplify_tol
        self.max_crowns_preview = max_crowns_preview
        self.crown_dir = crown_dir or self._resolve_dir('input/input_crowns', '../../input/input_crowns')
        self.ortho_dir = ortho_dir or self._resolve_dir('input/input_om', '../../input/input_om')
        
        self.file_pairs: List[Tuple[str, Optional[str]]] = []
        self.om_ids: List[int] = []
        self.crowns_gdfs: Dict[int, gpd.GeoDataFrame] = {}
        self.crown_attrs: Dict[int, List[Dict[str, Any]]] = {}
        self.crown_images: Dict[int, List[Optional[np.ndarray]]] = {}
        self.crown_crs: Dict[int, Optional[Any]] = {}
        self.ortho_crs: Dict[int, Optional[Any]] = {}
        self.G: nx.DiGraph = nx.DiGraph()
        self.match_case_mode: str = "balanced"
        
        # Use ultra-relaxed configs by default
        self.case_configs: Dict[str, MatchCaseConfig] = self._ultra_relaxed_case_configs()
        self.case_order: List[str] = ['one_to_one', 'containment', 'nearby']
        self.last_case_counts: Dict[str, int] = {}
        self.last_selected_counts: Dict[str, int] = {}
        
        if auto_discover:
            self.discover_files()
    
    @staticmethod
    def _resolve_dir(root_rel: str, nb_rel: str) -> str:
        """Resolve directory path from multiple possible locations."""
        candidates = [
            os.path.abspath(os.path.join(os.getcwd(), root_rel)),
            os.path.abspath(os.path.join(os.getcwd(), nb_rel)),
        ]
        for p in candidates:
            if os.path.isdir(p):
                return p
        raise FileNotFoundError(f"Could not resolve directory for {root_rel}. Tried: {candidates}")
    
    @staticmethod
    def _extract_numeric_id(name: str) -> Optional[int]:
        """Extract numeric ID from filename."""
        m = re.search(r"(\d+)", os.path.basename(name))
        return int(m.group(1)) if m else None
    
    def discover_files(self) -> None:
        """Automatically discover and pair crown GeoPackages with orthomosaics."""
        crown_files = [
            os.path.join(self.crown_dir, f) 
            for f in os.listdir(self.crown_dir) 
            if f.lower().endswith('.gpkg')
        ]
        ortho_files = [
            os.path.join(self.ortho_dir, f) 
            for f in os.listdir(self.ortho_dir) 
            if f.lower().endswith('.tif')
        ] if os.path.exists(self.ortho_dir) else []
        
        if not crown_files:
            raise FileNotFoundError(f"No .gpkg crown files found in {self.crown_dir}")
        
        # Group files by numeric ID
        crowns_by_id = {}
        for cf in crown_files:
            cid = self._extract_numeric_id(cf)
            crowns_by_id[cid if cid is not None else cf] = cf
        
        orthos_by_id = {}
        for of in ortho_files:
            oid = self._extract_numeric_id(of)
            orthos_by_id[oid if oid is not None else of] = of
        
        # Find matching numeric IDs
        numeric_ids = sorted(
            set(k for k in crowns_by_id.keys() if isinstance(k, int)) & 
            set(k for k in orthos_by_id.keys() if isinstance(k, int))
        )
        
        # Pair files
        file_pairs: List[Tuple[str, Optional[str]]] = []
        if numeric_ids:
            for nid in numeric_ids:
                file_pairs.append((crowns_by_id[nid], orthos_by_id.get(nid)))
            crown_only = sorted(
                k for k in crowns_by_id.keys() 
                if isinstance(k, int) and k not in numeric_ids
            )
            for nid in crown_only:
                file_pairs.append((crowns_by_id[nid], None))
        else:
            crown_files_sorted = sorted(crown_files)
            ortho_files_sorted = sorted(ortho_files)
            for i, cf in enumerate(crown_files_sorted):
                of = ortho_files_sorted[i] if i < len(ortho_files_sorted) else None
                file_pairs.append((cf, of))
        
        # Extract OM IDs
        om_ids: List[int] = []
        for cf, _ in file_pairs:
            cid = self._extract_numeric_id(cf)
            om_ids.append(cid if cid is not None else len(om_ids) + 1)
        
        # Sort by OM ID
        pairs_with_id = sorted(
            [(oid, cf, of) for oid, (cf, of) in zip(om_ids, file_pairs)],
            key=lambda x: x[0]
        )
        self.file_pairs = [(cf, of) for _, cf, of in pairs_with_id]
        self.om_ids = [oid for oid, _, _ in pairs_with_id]
    
    def load_data(
        self,
        load_images: bool = False,
        align: bool = False,
        reference_om_id: Optional[int] = None
    ) -> None:
        """
        Load crown data from GeoPackages and optionally extract image patches.
        
        CRITICAL: Images are extracted from ORIGINAL geometries BEFORE alignment.
        This ensures image-geometry correspondence is preserved.
        
        Args:
            load_images: If True, extract RGB image patches from orthomosaics for each crown
            align: If True, align all OMs to reference OM using translational shift
            reference_om_id: OM ID to use as alignment reference (default: first OM)
        """
        self.crowns_gdfs.clear()
        self.crown_attrs.clear()
        self.crown_images.clear()
        self.crown_crs.clear()
        self.ortho_crs.clear()
        
        print(f"\n{'='*70}")
        print("LOADING DATA")
        print(f"{'='*70}")
        
        # Step 1: Load geometries and extract images from ORIGINAL positions
        for om_id, (crown_file, ortho_file) in zip(self.om_ids, self.file_pairs):
            print(f"\nOM{om_id}: Loading {os.path.basename(crown_file)}")
            gdf = gpd.read_file(crown_file)
            self.crowns_gdfs[om_id] = gdf
            self.crown_crs[om_id] = gdf.crs
            
            # Extract images from ORIGINAL geometries BEFORE any shifting
            if load_images and ortho_file and os.path.exists(ortho_file):
                print(f"  Extracting {len(gdf)} crown images from {os.path.basename(ortho_file)}...")
                with rasterio.open(ortho_file) as src:
                    self.ortho_crs[om_id] = src.crs
                    patches: List[Optional[np.ndarray]] = []
                    for _, row in gdf.iterrows():
                        geom = [mapping(row.geometry)]
                        try:
                            out_image, _ = mask(src, geom, crop=True)
                            img_patch = np.moveaxis(out_image, 0, -1)  # (C, H, W) -> (H, W, C)
                        except Exception:
                            img_patch = None
                        patches.append(img_patch)
                self.crown_images[om_id] = patches
                print(f"  ✓ Extracted {sum(1 for p in patches if p is not None)} images")
            else:
                self.crown_images[om_id] = [None] * len(gdf)
                if ortho_file and os.path.exists(ortho_file):
                    with rasterio.open(ortho_file) as src:
                        self.ortho_crs[om_id] = src.crs
                else:
                    self.ortho_crs[om_id] = None
            
            # Compute attributes from original geometries
            self.crown_attrs[om_id] = [
                self._compute_crown_attributes(row.geometry) 
                for _, row in gdf.iterrows()
            ]
        
        self._check_crs_consistency()
        print(f"\n{'='*70}")
        
        # Step 2: Apply alignment if requested (geometries shift, images stay with original crowns)
        if align:
            print(f"\n{'='*70}")
            print("APPLYING SPATIAL ALIGNMENT")
            print(f"{'='*70}")
            self.alignment_shifts = self.align_to_reference(reference_om_id=reference_om_id)
            print(f"{'='*70}\n")
    
    def _check_crs_consistency(self) -> None:
        """Check CRS consistency across crown files and orthomosaics."""
        print("CRS CONSISTENCY CHECK")
        print("-" * 60)
        crown_crs_values = {om_id: crs for om_id, crs in self.crown_crs.items()}
        ortho_crs_values = {om_id: crs for om_id, crs in self.ortho_crs.items()}
        crown_crs_set = {str(crs) for crs in crown_crs_values.values() if crs is not None}
        if not crown_crs_set:
            print("⚠️  Crown CRS: None detected in all crown files")
        elif len(crown_crs_set) > 1:
            print("⚠️  Crown CRS mismatch across OMs:")
            for om_id, crs in crown_crs_values.items():
                print(f"  OM{om_id}: {crs}")
        else:
            print(f"✓ Crown CRS consistent: {next(iter(crown_crs_set))}")
        
        for om_id in self.om_ids:
            crown_crs = crown_crs_values.get(om_id)
            ortho_crs = ortho_crs_values.get(om_id)
            if crown_crs is None:
                print(f"⚠️  OM{om_id}: crown CRS is None")
            if ortho_crs is None:
                print(f"⚠️  OM{om_id}: ortho CRS is None or ortho missing")
            if crown_crs is not None and ortho_crs is not None and crown_crs != ortho_crs:
                print(f"⚠️  OM{om_id}: crown CRS != ortho CRS ({crown_crs} vs {ortho_crs})")
        print("-" * 60)
    
    def align_to_reference(
        self,
        reference_om_id: Optional[int] = None,
        threshold: float = 20.0,
        min_matches: int = 5
    ) -> Dict[int, Tuple[float, float]]:
        """
        Align all OMs to a reference OM using mutual nearest neighbors and robust median shift.
        
        Args:
            reference_om_id: OM ID to use as reference (default: first OM)
            threshold: Maximum distance for considering a match (meters)
            min_matches: Minimum number of matches required
        
        Returns:
            Dictionary mapping om_id -> (dx, dy) shift applied
        """
        if not self.crowns_gdfs:
            raise ValueError("No crown data loaded. Call load_data() first.")
        
        if reference_om_id is None:
            reference_om_id = self.om_ids[0]
        if reference_om_id not in self.crowns_gdfs:
            raise ValueError(f"Reference OM {reference_om_id} not found in loaded data.")
        
        ref = self.crowns_gdfs[reference_om_id]
        ref_centroids = np.array([[g.centroid.x, g.centroid.y] for g in ref.geometry])
        shifts = {reference_om_id: (0.0, 0.0)}
        
        print(f"Aligning all OMs to reference OM{reference_om_id}")
        print(f"  Reference crowns: {len(ref_centroids)}")
        print()
        
        nn_ref = NearestNeighbors(n_neighbors=1).fit(ref_centroids)

        for om_id in self.om_ids:
            if om_id == reference_om_id:
                continue
            curr = self.crowns_gdfs[om_id].copy()
            curr_centroids = np.array([[g.centroid.x, g.centroid.y] for g in curr.geometry])
            if len(curr_centroids) == 0:
                print(f"OM{om_id}: no crowns, skipping alignment")
                shifts[om_id] = (0.0, 0.0)
                continue
            
            dist_to_ref, idx_ref = nn_ref.kneighbors(curr_centroids)
            nn_curr = NearestNeighbors(n_neighbors=1).fit(curr_centroids)
            dist_to_curr, idx_curr = nn_curr.kneighbors(ref_centroids)
            
            mutual = []
            for i, (d, ref_idx) in enumerate(zip(dist_to_ref[:, 0], idx_ref[:, 0])):
                if d < threshold and idx_curr[ref_idx][0] == i and dist_to_curr[ref_idx][0] < threshold:
                    mutual.append((ref_idx, i, d))
            
            if len(mutual) < min_matches:
                relaxed = [(idx_ref[i][0], i, dist_to_ref[i][0]) for i in range(len(dist_to_ref)) if dist_to_ref[i][0] < threshold * 1.5]
                mutual = mutual + relaxed
            
            if len(mutual) < min_matches:
                print(f"OM{om_id}: insufficient matches ({len(mutual)}/{min_matches}), skipping alignment")
                shifts[om_id] = (0.0, 0.0)
                continue
            
            shifts_array = np.array([ref_centroids[a] - curr_centroids[b] for a, b, _ in mutual])
            median_shift = np.median(shifts_array, axis=0)
            residuals = np.linalg.norm(shifts_array - median_shift, axis=1)
            median_resid = float(np.median(residuals)) if len(residuals) else 0.0
            mad = float(np.median(np.abs(residuals - median_resid))) or 1e-6
            inliers = residuals <= median_resid + 3.0 * mad
            
            if not np.any(inliers):
                inliers = np.ones_like(residuals, dtype=bool)
            
            dx, dy = np.median(shifts_array[inliers], axis=0)
            print(f"OM{om_id}: matches={len(mutual)}, inliers={np.sum(inliers)}, dx={dx:.2f}, dy={dy:.2f}, med_res={median_resid:.2f}")
            
            curr['geometry'] = curr['geometry'].apply(lambda g: translate(g, xoff=dx, yoff=dy))
            self.crowns_gdfs[om_id] = curr
            self.crown_attrs[om_id] = [
                self._compute_crown_attributes(row.geometry) 
                for _, row in curr.iterrows()
            ]
            shifts[om_id] = (float(dx), float(dy))
        
        return shifts
    
    @staticmethod
    def _compute_crown_attributes(geometry) -> Dict[str, Any]:
        """Compute geometric attributes for a crown polygon."""
        centroid = geometry.centroid
        area = geometry.area
        perimeter = geometry.length
        bounds = geometry.bounds
        compactness = (4 * np.pi * area) / (perimeter ** 2) if perimeter > 0 else 0
        
        try:
            min_rect = geometry.minimum_rotated_rectangle
            coords = list(min_rect.exterior.coords)
            side1 = np.linalg.norm(np.array(coords[0]) - np.array(coords[1]))
            side2 = np.linalg.norm(np.array(coords[1]) - np.array(coords[2]))
            major_axis = max(side1, side2)
            minor_axis = min(side1, side2)
            eccentricity = minor_axis / major_axis if major_axis > 0 else 1
        except Exception:
            eccentricity = 1
        
        aspect_ratio = (bounds[3] - bounds[1]) / (bounds[2] - bounds[0]) if bounds[2] != bounds[0] else 1
        
        return {
            'geometry': geometry,
            'centroid': centroid,
            'area': area,
            'perimeter': perimeter,
            'compactness': compactness,
            'eccentricity': eccentricity,
            'aspect_ratio': aspect_ratio,
            'bounds': bounds,
        }
    
    def _ultra_relaxed_case_configs(self) -> Dict[str, MatchCaseConfig]:
        """
        ULTRA RELAXED matching parameters - prioritizes spatial proximity over shape similarity.
        
        Designed to bridge difficult OM transitions with minimal overlap or shape changes.
        """
        return {
            'one_to_one': MatchCaseConfig(
                name='one_to_one',
                base_similarity_weights={
                    'spatial': 0.50,  # Strong emphasis on proximity
                    'area': 0.15,
                    'shape': 0.05,    # De-emphasize shape
                    'iou': 0.30
                },
                scoring_weights={
                    'base': 0.60, 
                    'iou': 0.10, 
                    'overlap_prev': 0.05, 
                    'overlap_curr': 0.05, 
                    'centroid': 0.20
                },
                similarity_threshold=0.25,
                min_iou=0.01,
                min_overlap_prev=0.10,
                min_overlap_curr=0.10,
                max_centroid_dist=50.0,
                mutual_best=False,
                allow_multiple=True,
                max_edges_per_prev=3,
                max_edges_per_curr=3,
            ),
            'containment': MatchCaseConfig(
                name='containment',
                base_similarity_weights={
                    'spatial': 0.50,
                    'area': 0.20,
                    'shape': 0.10,
                    'iou': 0.20
                },
                scoring_weights={
                    'base': 0.40, 
                    'overlap_prev': 0.30, 
                    'overlap_curr': 0.30
                },
                similarity_threshold=0.25,
                min_iou=0.0,
                min_overlap_prev=0.25,
                min_overlap_curr=0.25,
                max_centroid_dist=150.0,
                mutual_best=False,
                allow_multiple=True,
                max_edges_per_prev=2,
                max_edges_per_curr=2,
            ),
            'nearby': MatchCaseConfig(
                name='nearby',
                base_similarity_weights={
                    'spatial': 0.85,  # Almost purely spatial
                    'area': 0.10,
                    'shape': 0.03,
                    'iou': 0.02
                },
                scoring_weights={
                    'base': 0.70, 
                    'centroid': 0.30
                },
                similarity_threshold=0.20,
                min_iou=0.0,
                min_overlap_prev=0.05,
                min_overlap_curr=0.05,
                max_centroid_dist=200.0,
                mutual_best=False,
                allow_multiple=True,
                max_edges_per_prev=2,
                max_edges_per_curr=2,
            ),
        }
    
    @staticmethod
    def _compute_iou(g1, g2) -> float:
        """Compute intersection over union for two geometries."""
        try:
            intersection = g1.intersection(g2).area
            union = g1.union(g2).area
        except Exception:
            intersection = 0.0
            union = g1.area + g2.area
        return intersection / union if union > 0 else 0.0
    
    def _weighted_similarity(
        self,
        a1: Dict[str, Any],
        a2: Dict[str, Any],
        weights: Optional[Dict[str, float]] = None,
        max_dist: float = 100.0
    ) -> Tuple[float, Dict[str, float]]:
        """Compute weighted similarity score between two crown attributes."""
        if weights is None:
            weights = {'spatial': 0.4, 'area': 0.2, 'shape': 0.2, 'iou': 0.2}
        
        # Spatial similarity
        centroid_dist = a1['centroid'].distance(a2['centroid'])
        spatial_sim = max(0.0, 1.0 - (centroid_dist / max_dist))
        
        # Area similarity
        area_sim = min(a1['area'], a2['area']) / max(a1['area'], a2['area']) if max(a1['area'], a2['area']) > 0 else 0.0
        
        # Shape similarity
        compactness_sim = 1.0 - abs(a1['compactness'] - a2['compactness'])
        eccentricity_sim = 1.0 - abs(a1['eccentricity'] - a2['eccentricity'])
        shape_sim = (compactness_sim + eccentricity_sim) / 2.0
        
        # IoU similarity
        iou_sim = self._compute_iou(a1['geometry'], a2['geometry'])
        
        # Weighted total
        total = (
            weights.get('spatial', 0.0) * spatial_sim +
            weights.get('area', 0.0) * area_sim +
            weights.get('shape', 0.0) * shape_sim +
            weights.get('iou', 0.0) * iou_sim
        )
        
        return total, {
            'spatial': float(spatial_sim),
            'area': float(area_sim),
            'shape': float(shape_sim),
            'iou': float(iou_sim),
            'total': float(total)
        }
    
    def _compute_pair_metrics(
        self,
        prev_attrs: Dict[str, Any],
        curr_attrs: Dict[str, Any],
        max_dist: float
    ) -> Dict[str, float]:
        """Compute comprehensive metrics for a crown pair."""
        prev_geom = prev_attrs['geometry']
        curr_geom = curr_attrs['geometry']
        
        try:
            intersection_area = prev_geom.intersection(curr_geom).area
        except Exception:
            intersection_area = 0.0
        
        try:
            union_area = prev_geom.union(curr_geom).area
        except Exception:
            union_area = prev_attrs['area'] + curr_attrs['area'] - intersection_area
        
        prev_area = prev_attrs['area'] if prev_attrs['area'] > 0 else 1e-6
        curr_area = curr_attrs['area'] if curr_attrs['area'] > 0 else 1e-6
        
        overlap_prev = intersection_area / prev_area
        overlap_curr = intersection_area / curr_area
        iou = intersection_area / union_area if union_area > 0 else 0.0
        
        centroid_dist = prev_attrs['centroid'].distance(curr_attrs['centroid'])
        base_similarity, parts = self._weighted_similarity(prev_attrs, curr_attrs, max_dist=max_dist)
        
        prev_radius = np.sqrt(prev_area / np.pi)
        curr_radius = np.sqrt(curr_area / np.pi)
        mean_radius = max((prev_radius + curr_radius) / 2.0, 1e-3)
        
        area_ratio = curr_area / prev_area if prev_area > 0 else np.inf
        if not np.isfinite(area_ratio) or area_ratio <= 0:
            balanced_area_ratio = 0.0
        else:
            balanced_area_ratio = area_ratio if area_ratio <= 1 else 1 / area_ratio
        
        try:
            prev_contains_curr = prev_geom.buffer(0).contains(curr_geom)
        except Exception:
            prev_contains_curr = False
        
        try:
            curr_contains_prev = curr_geom.buffer(0).contains(prev_geom)
        except Exception:
            curr_contains_prev = False
        
        return {
            'intersection_area': float(intersection_area),
            'union_area': float(union_area),
            'overlap_prev': float(overlap_prev),
            'overlap_curr': float(overlap_curr),
            'iou': float(iou),
            'centroid_dist': float(centroid_dist),
            'base_similarity': float(base_similarity),
            'spatial_similarity': float(parts['spatial']),
            'area_similarity': float(parts['area']),
            'shape_similarity': float(parts['shape']),
            'mean_radius': float(mean_radius),
            'area_ratio': float(area_ratio if np.isfinite(area_ratio) else 0.0),
            'balanced_area_ratio': float(balanced_area_ratio),
            'prev_contains_curr': bool(prev_contains_curr),
            'curr_contains_prev': bool(curr_contains_prev),
        }
    
    def _classify_match_case(
        self,
        prev_node: Tuple[int, int],
        curr_node: Tuple[int, int],
        features: Dict[str, float],
        prev_overlap_counts: Dict[Tuple[int, int], int],
        curr_overlap_counts: Dict[Tuple[int, int], int],
        overlap_gate: float,
        mode: str = "balanced"
    ) -> str:
        """Classify the type of match between two crowns."""
        if features['prev_contains_curr'] or features['curr_contains_prev']:
            return 'containment'
        
        overlap_prev = features['overlap_prev']
        overlap_curr = features['overlap_curr']
        iou = features['iou']
        centroid_dist = features['centroid_dist']
        prev_count = prev_overlap_counts.get(prev_node, 0)
        curr_count = curr_overlap_counts.get(curr_node, 0)
        
        if mode == "lite":
            min_overlap_one_to_one = max(overlap_gate, 0.10)
            min_iou_one_to_one = max(overlap_gate * 0.5, 0.04)
            if (overlap_prev >= min_overlap_one_to_one and
                overlap_curr >= min_overlap_one_to_one and
                iou >= min_iou_one_to_one and
                centroid_dist < 50.0):
                return 'one_to_one'
            near_gate = max(overlap_gate * 0.5, 0.01)
            if centroid_dist < 50.0 and (overlap_prev >= near_gate or overlap_curr >= near_gate):
                return 'nearby'
        elif mode == "strict":
            min_overlap_for_one_to_one = max(overlap_gate, 0.30)
            min_iou_for_one_to_one = max(overlap_gate / 2, 0.10)
            if (prev_count == 1 and curr_count == 1 and 
                overlap_prev >= min_overlap_for_one_to_one and 
                overlap_curr >= min_overlap_for_one_to_one and 
                iou >= min_iou_for_one_to_one):
                return 'one_to_one'
            if centroid_dist < 30.0 and (overlap_prev >= overlap_gate or overlap_curr >= overlap_gate):
                return 'nearby'
        else:
            min_overlap_one_to_one = max(overlap_gate, 0.15)
            min_iou_one_to_one = max(overlap_gate * 0.5, 0.05)
            if (overlap_prev >= min_overlap_one_to_one and
                overlap_curr >= min_overlap_one_to_one and
                iou >= min_iou_one_to_one and
                centroid_dist < 40.0):
                return 'one_to_one'
            near_gate = max(overlap_gate * 0.5, 0.02)
            if centroid_dist < 35.0 and (overlap_prev >= near_gate or overlap_curr >= near_gate):
                return 'nearby'
        
        return 'none'
    
    def _score_candidate(
        self,
        base_similarity: float,
        similarity_parts: Dict[str, float],
        features: Dict[str, float],
        config: MatchCaseConfig
    ) -> float:
        """Score a candidate match using weighted components."""
        centroid_factor = 1.0 - min(1.0, features['centroid_dist'] / (features['mean_radius'] * 3.0))
        
        components = {
            'base': base_similarity,
            'spatial': similarity_parts.get('spatial', 0.0),
            'area': similarity_parts.get('area', 0.0),
            'shape': similarity_parts.get('shape', 0.0),
            'iou': features['iou'],
            'overlap_prev': features['overlap_prev'],
            'overlap_curr': features['overlap_curr'],
            'centroid': max(0.0, centroid_factor),
            'area_balance': features.get('balanced_area_ratio', 0.0),
        }
        
        score = 0.0
        for key, weight in config.scoring_weights.items():
            score += weight * components.get(key, 0.0)
        
        return score
    
    def _select_candidates_by_case(
        self,
        candidates: List[Dict[str, Any]],
        configs: Dict[str, MatchCaseConfig],
        case_order: List[str],
        max_dist: float,
        min_base_similarity: float = 0.0
    ) -> List[Dict[str, Any]]:
        """Select best candidates for each case in priority order."""
        selected: List[Dict[str, Any]] = []
        used_prev: Dict[Tuple[int, int], int] = defaultdict(int)
        used_curr: Dict[Tuple[int, int], int] = defaultdict(int)
        
        for case_name in case_order:
            config = configs.get(case_name)
            if not config:
                continue
            
            case_candidates = [cand for cand in candidates if cand['case'] == case_name]
            if not case_candidates:
                continue
            
            # Enrich candidates with scores
            enriched: List[Dict[str, Any]] = []
            for cand in case_candidates:
                prev_attrs = cand['prev_attrs']
                curr_attrs = cand['curr_attrs']
                features = cand['features']
                
                # Apply filters
                if config.max_centroid_dist is not None and features['centroid_dist'] > config.max_centroid_dist:
                    continue
                if features['iou'] < config.min_iou:
                    continue
                if features['overlap_prev'] < config.min_overlap_prev:
                    continue
                if features['overlap_curr'] < config.min_overlap_curr:
                    continue
                
                # Compute score (case-specific weights)
                base_similarity, parts = self._weighted_similarity(
                    prev_attrs, curr_attrs,
                    weights=config.base_similarity_weights,
                    max_dist=max_dist
                )
                if min_base_similarity and base_similarity < min_base_similarity:
                    continue
                score = self._score_candidate(base_similarity, parts, features, config)
                
                if score < config.similarity_threshold:
                    continue
                
                cand['base_similarity'] = float(base_similarity)
                cand['similarity_parts'] = {k: float(v) for k, v in parts.items()}
                cand['score'] = float(score)
                enriched.append(cand)
            
            if not enriched:
                continue
            
            # Select candidates (allow multiple edges when configured)
            if config.mutual_best:
                best_prev: Dict[Tuple[int, int], Dict[str, Any]] = {}
                best_curr: Dict[Tuple[int, int], Dict[str, Any]] = {}
                
                for cand in enriched:
                    prev_node = cand['prev_node']
                    curr_node = cand['curr_node']
                    
                    if not config.allow_multiple:
                        if used_prev.get(prev_node, 0) or used_curr.get(curr_node, 0):
                            continue
                    
                    if prev_node not in best_prev or cand['score'] > best_prev[prev_node]['score']:
                        best_prev[prev_node] = cand
                    if curr_node not in best_curr or cand['score'] > best_curr[curr_node]['score']:
                        best_curr[curr_node] = cand
                
                for cand in enriched:
                    prev_node = cand['prev_node']
                    curr_node = cand['curr_node']
                    
                    if best_prev.get(prev_node) is cand and best_curr.get(curr_node) is cand:
                        if not config.allow_multiple:
                            if used_prev.get(prev_node, 0) or used_curr.get(curr_node, 0):
                                continue
                        
                        if config.max_edges_per_prev is not None and used_prev[prev_node] >= config.max_edges_per_prev:
                            continue
                        if config.max_edges_per_curr is not None and used_curr[curr_node] >= config.max_edges_per_curr:
                            continue
                        
                        selected.append(cand)
                        used_prev[prev_node] += 1
                        used_curr[curr_node] += 1
            else:
                enriched.sort(key=lambda c: c['score'], reverse=True)
                
                for cand in enriched:
                    prev_node = cand['prev_node']
                    curr_node = cand['curr_node']
                    
                    if not config.allow_multiple:
                        if used_prev.get(prev_node, 0) or used_curr.get(curr_node, 0):
                            continue
                    
                    if config.max_edges_per_prev is not None and used_prev[prev_node] >= config.max_edges_per_prev:
                        continue
                    if config.max_edges_per_curr is not None and used_curr[curr_node] >= config.max_edges_per_curr:
                        continue
                    
                    selected.append(cand)
                    used_prev[prev_node] += 1
                    used_curr[curr_node] += 1
        
        return selected
    
    def reset_graph(self) -> None:
        """Clear the graph."""
        self.G = nx.DiGraph()
    
    def build_graph_conditional(
        self,
        case_configs: Optional[Dict[str, MatchCaseConfig]] = None,
        case_order: Optional[List[str]] = None,
        base_max_dist: float = 200.0,
        overlap_gate: float = 0.05,
        min_base_similarity: float = 0.0,
        max_candidates_per_prev: Optional[int] = None,
        max_candidates_per_curr: Optional[int] = None,
        classify_mode: Optional[str] = None
    ) -> None:
        """
        Build tracking graph using conditional case-based matching.
        
        Args:
            case_configs: Dictionary of case configurations (default: ultra-relaxed)
            case_order: Priority order for cases (default: ['one_to_one', 'containment', 'nearby'])
            base_max_dist: Maximum centroid distance for candidate matches (meters)
            overlap_gate: Minimum overlap fraction to count as overlapping
            min_base_similarity: Minimum base similarity (case-specific weights) for candidate filtering
            max_candidates_per_prev: Maximum candidates to consider per previous crown
            max_candidates_per_curr: Maximum candidates to consider per current crown
            classify_mode: one of {'balanced','lite','strict'} (default: self.match_case_mode)
        """
        if not self.crowns_gdfs:
            self.load_data(load_images=False)
        
        self.reset_graph()
        
        configs = {name: replace(cfg) for name, cfg in (case_configs or self.case_configs).items()}
        order = case_order or self.case_order
        mode = classify_mode or self.match_case_mode
        
        self.last_case_counts = {}
        self.last_selected_counts = {name: 0 for name in configs.keys()}
        
        print(f"\n{'='*70}")
        print("BUILDING TRACKING GRAPH")
        print(f"{'='*70}")
        print(f"Parameters:")
        print(f"  base_max_dist: {base_max_dist}m")
        print(f"  overlap_gate: {overlap_gate}")
        print(f"  min_base_similarity: {min_base_similarity}")
        print(f"  classify_mode: {mode}")
        print(f"  case_order: {', '.join(order)}")
        print()
        
        # Add all nodes
        for idx in range(len(self.om_ids)):
            om_id = self.om_ids[idx]
            gdf = self.crowns_gdfs[om_id]
            
            for crown_id, row in gdf.iterrows():
                attrs = self.crown_attrs[om_id][crown_id]
                self.G.add_node((om_id, crown_id), **attrs)
            
            if idx == 0:
                continue
            
            # Build edges between consecutive OMs
            prev_om = self.om_ids[idx - 1]
            prev_nodes = [(prev_om, i) for i in range(len(self.crowns_gdfs[prev_om]))]
            curr_nodes = [(om_id, j) for j in range(len(gdf))]
            
            print(f"OM{prev_om} → OM{om_id}: {len(prev_nodes)} × {len(curr_nodes)} potential pairs")
            
            # Generate candidates
            candidates: List[Dict[str, Any]] = []
            overlap_counts_prev: Dict[Tuple[int, int], int] = defaultdict(int)
            overlap_counts_curr: Dict[Tuple[int, int], int] = defaultdict(int)
            
            for prev_node in prev_nodes:
                prev_attrs = self.G.nodes[prev_node]
                
                for curr_node in curr_nodes:
                    curr_attrs = self.crown_attrs[om_id][curr_node[1]]
                    
                    features = self._compute_pair_metrics(prev_attrs, curr_attrs, max_dist=base_max_dist)
                    
                    # Early filtering (distance only)
                    if features['centroid_dist'] > base_max_dist:
                        continue
                    
                    cand = {
                        'prev_node': prev_node,
                        'curr_node': curr_node,
                        'prev_attrs': prev_attrs,
                        'curr_attrs': curr_attrs,
                        'features': features,
                    }
                    candidates.append(cand)
                    
                    if features['overlap_prev'] >= overlap_gate:
                        overlap_counts_prev[prev_node] += 1
                    if features['overlap_curr'] >= overlap_gate:
                        overlap_counts_curr[curr_node] += 1
            
            if not candidates:
                print(f"  No candidates found")
                continue
            
            # Classify candidates
            for cand in candidates:
                cand['case'] = self._classify_match_case(
                    cand['prev_node'], cand['curr_node'], cand['features'],
                    overlap_counts_prev, overlap_counts_curr, overlap_gate, mode=mode
                )
            
            candidates = [cand for cand in candidates if cand['case'] != 'none']
            
            if not candidates:
                print(f"  No valid cases found")
                continue
            
            # Trim candidates if requested
            if max_candidates_per_prev is not None:
                grouped_prev: Dict[Tuple[int, int], List[Dict[str, Any]]] = defaultdict(list)
                for cand in candidates:
                    grouped_prev[cand['prev_node']].append(cand)
                
                trimmed: List[Dict[str, Any]] = []
                for group in grouped_prev.values():
                    group.sort(key=lambda c: (-c['features']['iou'], c['features']['centroid_dist']))
                    trimmed.extend(group[:max_candidates_per_prev])
                candidates = trimmed
            
            if max_candidates_per_curr is not None:
                grouped_curr: Dict[Tuple[int, int], List[Dict[str, Any]]] = defaultdict(list)
                for cand in candidates:
                    grouped_curr[cand['curr_node']].append(cand)
                
                trimmed_curr: List[Dict[str, Any]] = []
                for group in grouped_curr.values():
                    group.sort(key=lambda c: (-c['features']['iou'], c['features']['centroid_dist']))
                    trimmed_curr.extend(group[:max_candidates_per_curr])
                candidates = trimmed_curr
            
            # Count candidates by case
            case_counts = defaultdict(int)
            for cand in candidates:
                case_counts[cand['case']] += 1
            
            for case_name, count in case_counts.items():
                self.last_case_counts[case_name] = self.last_case_counts.get(case_name, 0) + count
            
            print(f"  Candidates by case: {dict(case_counts)}")
            
            # Select best candidates
            selected = self._select_candidates_by_case(
                candidates, configs, order, base_max_dist, min_base_similarity=min_base_similarity
            )
            
            # Add edges to graph
            for cand in selected:
                case_name = cand['case']
                features = cand['features']
                similarity_parts = cand.get('similarity_parts', {})
                
                self.G.add_edge(
                    cand['prev_node'],
                    cand['curr_node'],
                    similarity=float(cand.get('score', features['base_similarity'])),
                    method='conditional',
                    case=case_name,
                    overlap_prev=float(features['overlap_prev']),
                    overlap_curr=float(features['overlap_curr']),
                    iou=float(features['iou']),
                    centroid_distance=float(features['centroid_dist']),
                    base_similarity=float(cand.get('base_similarity', features['base_similarity'])),
                    spatial_similarity=float(similarity_parts.get('spatial', features['spatial_similarity'])),
                    area_similarity=float(similarity_parts.get('area', features['area_similarity'])),
                    shape_similarity=float(similarity_parts.get('shape', features['shape_similarity']))
                )
                
                self.last_selected_counts[case_name] = self.last_selected_counts.get(case_name, 0) + 1
            
            print(f"  Selected: {len(selected)} edges")
            print()
        
        print(f"{'='*70}")
        print(f"Graph built: {self.G.number_of_nodes()} nodes, {self.G.number_of_edges()} edges")
        print(f"{'='*70}\n")
    
    def debug_candidate_flow(
        self,
        base_max_dist: float,
        overlap_gate: float,
        min_base_similarity: float = 0.0,
        case_configs: Optional[Dict[str, MatchCaseConfig]] = None,
        case_order: Optional[List[str]] = None,
        classify_mode: Optional[str] = None,
        max_candidates_per_prev: Optional[int] = None,
        max_candidates_per_curr: Optional[int] = None
    ) -> Dict[str, Any]:
        """
        Diagnose candidate filtering stages and report potential early-discard risks.
        """
        if not self.crowns_gdfs:
            raise ValueError("No crown data loaded. Call load_data() first.")
        configs = {name: replace(cfg) for name, cfg in (case_configs or self.case_configs).items()}
        order = case_order or self.case_order
        mode = classify_mode or self.match_case_mode
        
        summary: Dict[str, Any] = {
            'pairs': {},
            'legacy_prefilter_discards': 0,
            'legacy_prefilter_discards_that_would_pass': 0
        }
        
        for idx in range(1, len(self.om_ids)):
            prev_om = self.om_ids[idx - 1]
            curr_om = self.om_ids[idx]
            prev_nodes = [(prev_om, i) for i in range(len(self.crowns_gdfs[prev_om]))]
            curr_nodes = [(curr_om, j) for j in range(len(self.crowns_gdfs[curr_om]))]
            total_pairs = len(prev_nodes) * len(curr_nodes)
            
            candidates = []
            overlap_counts_prev: Dict[Tuple[int, int], int] = defaultdict(int)
            overlap_counts_curr: Dict[Tuple[int, int], int] = defaultdict(int)
            
            dist_pass = 0
            for prev_node in prev_nodes:
                prev_attrs = self.crown_attrs[prev_om][prev_node[1]]
                for curr_node in curr_nodes:
                    curr_attrs = self.crown_attrs[curr_om][curr_node[1]]
                    features = self._compute_pair_metrics(prev_attrs, curr_attrs, max_dist=base_max_dist)
                    if features['centroid_dist'] > base_max_dist:
                        continue
                    dist_pass += 1
                    cand = {
                        'prev_node': prev_node,
                        'curr_node': curr_node,
                        'prev_attrs': prev_attrs,
                        'curr_attrs': curr_attrs,
                        'features': features,
                    }
                    candidates.append(cand)
                    if features['overlap_prev'] >= overlap_gate:
                        overlap_counts_prev[prev_node] += 1
                    if features['overlap_curr'] >= overlap_gate:
                        overlap_counts_curr[curr_node] += 1
            
            for cand in candidates:
                cand['case'] = self._classify_match_case(
                    cand['prev_node'], cand['curr_node'], cand['features'],
                    overlap_counts_prev, overlap_counts_curr, overlap_gate, mode=mode
                )
            classified = [c for c in candidates if c['case'] != 'none']
            
            # Trim candidates if requested
            if max_candidates_per_prev is not None:
                grouped_prev: Dict[Tuple[int, int], List[Dict[str, Any]]] = defaultdict(list)
                for cand in classified:
                    grouped_prev[cand['prev_node']].append(cand)
                trimmed: List[Dict[str, Any]] = []
                for group in grouped_prev.values():
                    group.sort(key=lambda c: (-c['features']['iou'], c['features']['centroid_dist']))
                    trimmed.extend(group[:max_candidates_per_prev])
                classified = trimmed
            
            if max_candidates_per_curr is not None:
                grouped_curr: Dict[Tuple[int, int], List[Dict[str, Any]]] = defaultdict(list)
                for cand in classified:
                    grouped_curr[cand['curr_node']].append(cand)
                trimmed_curr: List[Dict[str, Any]] = []
                for group in grouped_curr.values():
                    group.sort(key=lambda c: (-c['features']['iou'], c['features']['centroid_dist']))
                    trimmed_curr.extend(group[:max_candidates_per_curr])
                classified = trimmed_curr
            
            # Score with case-specific weights
            scored = 0
            for cand in classified:
                cfg = configs.get(cand['case'])
                if not cfg:
                    continue
                features = cand['features']
                if cfg.max_centroid_dist is not None and features['centroid_dist'] > cfg.max_centroid_dist:
                    continue
                if features['iou'] < cfg.min_iou:
                    continue
                if features['overlap_prev'] < cfg.min_overlap_prev:
                    continue
                if features['overlap_curr'] < cfg.min_overlap_curr:
                    continue
                base_similarity, parts = self._weighted_similarity(
                    cand['prev_attrs'], cand['curr_attrs'],
                    weights=cfg.base_similarity_weights, max_dist=base_max_dist
                )
                if min_base_similarity and base_similarity < min_base_similarity:
                    continue
                score = self._score_candidate(base_similarity, parts, features, cfg)
                if score < cfg.similarity_threshold:
                    continue
                scored += 1
                
                # legacy prefilter test (old behavior)
                legacy_reject = features['base_similarity'] < min_base_similarity and features['iou'] < overlap_gate
                if legacy_reject:
                    summary['legacy_prefilter_discards'] += 1
                    summary['legacy_prefilter_discards_that_would_pass'] += 1
            
            summary['pairs'][f"{prev_om}->{curr_om}"] = {
                'total_pairs': total_pairs,
                'within_distance': dist_pass,
                'classified_candidates': len(classified),
                'case_scored_pass': scored,
            }
        
        return summary
    
    def _extract_all_chains(self) -> List[List[Tuple[int, int]]]:
        """Extract all tracking chains from the graph."""
        visited = set()
        chains: List[List[Tuple[int, int]]] = []
        
        # Start from nodes with no predecessors
        chain_starts = [n for n in self.G.nodes if not list(self.G.predecessors(n))]
        
        for start_node in chain_starts:
            if start_node in visited:
                continue
            
            chain = self._greedy_chain(start_node)
            chains.append(chain)
            visited.update(chain)
        
        # Handle remaining nodes (cycles or isolated)
        remaining = set(self.G.nodes) - visited
        for node in remaining:
            chains.append([node])
        
        return chains
    
    def _greedy_chain(self, start_node: Tuple[int, int]) -> List[Tuple[int, int]]:
        """Extract a single chain starting from a given node."""
        chain = [start_node]
        current = start_node
        
        while True:
            successors = list(self.G.successors(current))
            if not successors:
                break
            
            if len(successors) > 1:
                # Choose best successor
                best_successor = max(
                    successors,
                    key=lambda n: self.G[current][n].get('similarity', 0.0)
                )
                chain.append(best_successor)
                current = best_successor
            else:
                chain.append(successors[0])
                current = successors[0]
        
        return chain
    
    def quality_report(self) -> Tuple[str, Dict[str, Any]]:
        """Generate quality metrics and report."""
        G = self.G
        om_ids = self.om_ids
        
        metrics: Dict[str, Any] = {
            'total_trees_detected': G.number_of_nodes(),
            'total_edges': G.number_of_edges(),
            'total_possible_matches': 0,
            'successful_matches': 0,
            'match_rate_by_om_pair': {},
            'chain_length_distribution': {},
            'average_chain_length': 0,
            'median_chain_length': 0,
            'max_chain_length': 0,
        }
        
        # Chain analysis
        chains = self._extract_all_chains()
        chain_lengths = [len(chain) for chain in chains]
        
        if chain_lengths:
            metrics['average_chain_length'] = float(np.mean(chain_lengths))
            metrics['median_chain_length'] = float(np.median(chain_lengths))
            metrics['max_chain_length'] = int(max(chain_lengths))
            
            for length in chain_lengths:
                metrics['chain_length_distribution'][int(length)] = metrics['chain_length_distribution'].get(int(length), 0) + 1
        
        # Match rate by OM pair
        for i in range(len(om_ids) - 1):
            om1, om2 = om_ids[i], om_ids[i + 1]
            om1_nodes = [n for n in G.nodes if n[0] == om1]
            om2_nodes = [n for n in G.nodes if n[0] == om2]
            
            matches = sum(1 for u, v in G.edges() if u[0] == om1 and v[0] == om2)
            possible_matches = min(len(om1_nodes), len(om2_nodes))
            match_rate = matches / possible_matches if possible_matches > 0 else 0.0
            
            metrics['match_rate_by_om_pair'][f"{om1}->{om2}"] = {
                'matches': matches,
                'possible': possible_matches,
                'rate': float(match_rate),
            }
            
            metrics['total_possible_matches'] += possible_matches
            metrics['successful_matches'] += matches
        
        metrics['overall_match_rate'] = (
            metrics['successful_matches'] / metrics['total_possible_matches']
            if metrics['total_possible_matches'] > 0 else 0.0
        )
        
        # Generate report
        report = [
            "# Tree Tracking Quality Assessment Report",
            f"Total Trees Detected: {metrics['total_trees_detected']}",
            f"Total Tracking Edges: {metrics['total_edges']}",
            f"Overall Match Rate: {metrics['overall_match_rate']:.3f}",
            f"Average Chain Length: {metrics.get('average_chain_length', 0):.2f}",
            f"Maximum Chain Length: {metrics.get('max_chain_length', 0)}",
            "\nMatch Rates by Orthomosaic Pair:",
        ]
        
        for pair, data in metrics['match_rate_by_om_pair'].items():
            report.append(f"- {pair}: {data['matches']}/{data['possible']} ({data['rate']:.3f})")
        
        report.append("\nChain Length Distribution:")
        for length, count in sorted(metrics['chain_length_distribution'].items()):
            report.append(f"- Length {length}: {count} trees")
        
        if self.last_selected_counts:
            report.append("\nEdge selection by case:")
            for case_name, count in sorted(self.last_selected_counts.items(), key=lambda kv: (-kv[1], kv[0])):
                total_candidates = self.last_case_counts.get(case_name, 0)
                if total_candidates:
                    ratio = count / total_candidates
                    report.append(f"- {case_name}: {count} / {total_candidates} ({ratio:.2f})")
                else:
                    report.append(f"- {case_name}: {count}")
        
        return "\n".join(report), metrics
    
    def get_matching_chain(self, start_om_id: int, crown_id: int) -> List[Tuple[int, int]]:
        """Get the tracking chain starting from a specific crown."""
        node = (start_om_id, crown_id)
        if node not in self.G:
            raise ValueError(f"Node {(start_om_id, crown_id)} not in graph. Build the graph first.")
        return self._greedy_chain(node)
    
    def ensure_output_dir(self) -> None:
        """Create output directory if it doesn't exist."""
        os.makedirs(self.output_dir, exist_ok=True)
    
    def save_text(self, text: str, filename: str) -> str:
        """Save text to file in output directory."""
        self.ensure_output_dir()
        path = os.path.join(self.output_dir, filename)
        with open(path, 'w') as f:
            f.write(text)
        return path
    
    def save_json(self, data: Dict[str, Any], filename: str) -> str:
        """Save JSON data to file in output directory."""
        self.ensure_output_dir()
        path = os.path.join(self.output_dir, filename)
        with open(path, 'w') as f:
            json.dump(data, f, indent=2)
        return path

print("✓ TreeTrackingGraph class defined with configurable alignment and match modes")

In [ ]:
# ---- Phase-correlation alignment (orthomosaics) ----
def _read_ortho_lowres(path: str, max_size: int = 1200):
    with rasterio.open(path) as src:
        scale = max(src.width, src.height) / max_size if max(src.width, src.height) > max_size else 1.0
        out_w = int(round(src.width / scale))
        out_h = int(round(src.height / scale))
        data = src.read([1, 2, 3], out_shape=(3, out_h, out_w), resampling=rasterio.enums.Resampling.bilinear)
        rgb = np.moveaxis(data, 0, -1).astype(np.float32)
        rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-6)
        gray = 0.2989 * rgb[..., 0] + 0.5870 * rgb[..., 1] + 0.1140 * rgb[..., 2]
        scale_x = src.width / out_w
        scale_y = src.height / out_h
        transform = src.transform * rasterio.transform.Affine.scale(scale_x, scale_y)
        return rgb, gray, transform

def _bounds_from_transform(h: int, w: int, transform: rasterio.transform.Affine):
    corners = [(0, 0), (0, w), (h, 0), (h, w)]
    xs = []
    ys = []
    for row, col in corners:
        x, y = rasterio.transform.xy(transform, row, col, offset='ul')
        xs.append(x)
        ys.append(y)
    left, right = min(xs), max(xs)
    bottom, top = min(ys), max(ys)
    return left, bottom, right, top

def _safe_window(win: rasterio.windows.Window, height: int, width: int) -> rasterio.windows.Window:
    col_off = int(max(0, min(width, win.col_off)))
    row_off = int(max(0, min(height, win.row_off)))
    max_w = max(0, width - col_off)
    max_h = max(0, height - row_off)
    w = int(max(0, min(max_w, win.width)))
    h = int(max(0, min(max_h, win.height)))
    return rasterio.windows.Window(col_off=col_off, row_off=row_off, width=w, height=h)

def _crop_to_overlap(ref_img, mov_img, ref_transform, mov_transform):
    ref_h, ref_w = ref_img.shape[:2]
    mov_h, mov_w = mov_img.shape[:2]
    ref_left, ref_bottom, ref_right, ref_top = _bounds_from_transform(ref_h, ref_w, ref_transform)
    mov_left, mov_bottom, mov_right, mov_top = _bounds_from_transform(mov_h, mov_w, mov_transform)
    left = max(ref_left, mov_left)
    right = min(ref_right, mov_right)
    bottom = max(ref_bottom, mov_bottom)
    top = min(ref_top, mov_top)
    if left >= right or bottom >= top:
        return None, None
    ref_win = rasterio.windows.from_bounds(left, bottom, right, top, ref_transform).round_offsets().round_lengths()
    mov_win = rasterio.windows.from_bounds(left, bottom, right, top, mov_transform).round_offsets().round_lengths()
    ref_win = _safe_window(ref_win, ref_h, ref_w)
    mov_win = _safe_window(mov_win, mov_h, mov_w)
    if ref_win.width < 2 or ref_win.height < 2 or mov_win.width < 2 or mov_win.height < 2:
        return None, None
    ref_crop = ref_img[int(ref_win.row_off): int(ref_win.row_off + ref_win.height),
                       int(ref_win.col_off): int(ref_win.col_off + ref_win.width)]
    mov_crop = mov_img[int(mov_win.row_off): int(mov_win.row_off + mov_win.height),
                       int(mov_win.col_off): int(mov_win.col_off + mov_win.width)]
    return ref_crop, mov_crop

def _match_shape(ref_crop, mov_crop):
    if ref_crop.shape == mov_crop.shape:
        return ref_crop, mov_crop
    if not SKIMAGE_AVAILABLE:
        min_h = min(ref_crop.shape[0], mov_crop.shape[0])
        min_w = min(ref_crop.shape[1], mov_crop.shape[1])
        return ref_crop[:min_h, :min_w], mov_crop[:min_h, :min_w]
    mov_resized = resize(mov_crop, ref_crop.shape, preserve_range=True, anti_aliasing=True)
    return ref_crop, mov_resized

def _phase_corr_shift(ref_gray, mov_gray, ref_transform, mov_transform):
    ref_crop, mov_crop = _crop_to_overlap(ref_gray, mov_gray, ref_transform, mov_transform)
    if ref_crop is None or mov_crop is None:
        return None, None
    ref_crop, mov_crop = _match_shape(ref_crop, mov_crop)
    shift, error, _ = phase_cross_correlation(ref_crop, mov_crop, upsample_factor=10)
    shift_row, shift_col = shift
    dx = shift_col * ref_transform.a
    dy = shift_row * ref_transform.e
    return (float(dx), float(dy)), {'shift_px': (float(shift_row), float(shift_col)), 'error': float(error)}

def phase_corr_align_tracker(tracker: 'TreeTrackingGraph', reference_om_id: Optional[int] = None, max_preview: int = 1200):
    if not SKIMAGE_AVAILABLE:
        raise RuntimeError('scikit-image not available; cannot run phase correlation alignment.')
    if reference_om_id is None:
        reference_om_id = tracker.om_ids[0]
    # Load low-res orthos
    ortho_gray = {}
    ortho_transform = {}
    for om_id, (_, ortho_file) in zip(tracker.om_ids, tracker.file_pairs):
        if ortho_file and os.path.exists(ortho_file):
            _, gray, transform = _read_ortho_lowres(ortho_file, max_preview)
            ortho_gray[om_id] = gray
            ortho_transform[om_id] = transform
        else:
            ortho_gray[om_id] = None
            ortho_transform[om_id] = None
    shifts = {reference_om_id: (0.0, 0.0)}
    # Build cumulative shifts across consecutive OMs
    for idx in range(1, len(tracker.om_ids)):
        prev_id = tracker.om_ids[idx - 1]
        curr_id = tracker.om_ids[idx]
        ref_gray = ortho_gray.get(prev_id)
        mov_gray = ortho_gray.get(curr_id)
        ref_transform = ortho_transform.get(prev_id)
        mov_transform = ortho_transform.get(curr_id)
        if ref_gray is None or mov_gray is None or ref_transform is None or mov_transform is None:
            shifts[curr_id] = shifts.get(prev_id, (0.0, 0.0))
            continue
        out = _phase_corr_shift(ref_gray, mov_gray, ref_transform, mov_transform)
        if out[0] is None:
            shifts[curr_id] = shifts.get(prev_id, (0.0, 0.0))
            continue
        (dx, dy), meta = out
        prev_dx, prev_dy = shifts.get(prev_id, (0.0, 0.0))
        shifts[curr_id] = (prev_dx + dx, prev_dy + dy)
    # Apply shifts
    for om_id in tracker.om_ids:
        if om_id == reference_om_id:
            continue
        dx, dy = shifts.get(om_id, (0.0, 0.0))
        gdf = tracker.crowns_gdfs[om_id].copy()
        gdf['geometry'] = gdf['geometry'].apply(lambda g: translate(g, xoff=dx, yoff=dy))
        tracker.crowns_gdfs[om_id] = gdf
        tracker.crown_attrs[om_id] = [tracker._compute_crown_attributes(row.geometry) for _, row in gdf.iterrows()]
    tracker.alignment_shifts = shifts
    return shifts

In [ ]:
# ---- Multithreshold crowns + conservative chain completion ----
# Median-centroid alignment disabled; PCC only.
def _align_to_reference_pcc(self, reference_om_id=None, threshold=None, min_matches=None, max_preview=1200):
    print("Using phase cross-correlation alignment (median-centroid alignment disabled).")
    return phase_corr_align_tracker(self, reference_om_id=reference_om_id, max_preview=max_preview)
TreeTrackingGraph.align_to_reference = _align_to_reference_pcc

import pandas as pd
try:
    import fiona
    FIONA_AVAILABLE = True
except Exception as e:
    FIONA_AVAILABLE = False
    print("⚠️ fiona not available. Multithreshold layer listing may fail.")
    print(str(e))

def _threshold_tag_to_float(tag: str) -> float:
    try:
        return float(tag.replace("conf_", "").replace("p", "."))
    except Exception:
        return float("nan")

def _float_to_threshold_tag(val: float) -> str:
    return f"conf_{val:.2f}".replace(".", "p")

def _list_threshold_layers(gpkg_path: str) -> List[str]:
    meta_path = os.path.splitext(gpkg_path)[0] + ".meta.json"
    layers = []
    if os.path.exists(meta_path):
        try:
            with open(meta_path, "r") as f:
                meta = json.load(f)
            raw_thresholds = meta.get("thresholds", [])
            for t in raw_thresholds:
                if isinstance(t, str) and t.startswith("conf_"):
                    layers.append(t)
                elif isinstance(t, (int, float)):
                    layers.append(_float_to_threshold_tag(float(t)))
            layers = list(dict.fromkeys(layers))
        except Exception as e:
            print(f"⚠️ Failed reading meta: {meta_path} ({e})")
    if not layers:
        if not FIONA_AVAILABLE:
            raise RuntimeError("No meta layers and fiona unavailable. Cannot list GPKG layers.")
        layers = fiona.listlayers(gpkg_path)
    return sorted(layers, key=_threshold_tag_to_float)

def _load_layer_gdf(gpkg_path: str, layer_tag: str) -> gpd.GeoDataFrame:
    gdf = gpd.read_file(gpkg_path, layer=layer_tag)
    if 'threshold_tag' not in gdf.columns:
        gdf['threshold_tag'] = layer_tag
    if 'confidence' not in gdf.columns:
        gdf['confidence'] = np.nan
    if 'orthomosaic' not in gdf.columns:
        gdf['orthomosaic'] = os.path.basename(gpkg_path)
    if 'is_augmented' not in gdf.columns:
        gdf['is_augmented'] = False
    return gdf

def _align_gdf_to_tracker(gdf: gpd.GeoDataFrame, om_id: int, tracker: 'TreeTrackingGraph') -> gpd.GeoDataFrame:
    dx, dy = tracker.alignment_shifts.get(om_id, (0.0, 0.0))
    gdf_aligned = gdf.copy()
    gdf_aligned['geometry'] = gdf_aligned['geometry'].apply(lambda g: translate(g, xoff=dx, yoff=dy))
    return gdf_aligned

def load_multithreshold_base_layer(
    tracker: 'TreeTrackingGraph',
    base_threshold_tag: Optional[str] = None,
    load_images: bool = False
):
    tracker.crowns_gdfs.clear()
    tracker.crown_attrs.clear()
    tracker.crown_images.clear()
    tracker.crown_crs.clear()
    tracker.ortho_crs.clear()
    tracker.multithreshold_layers = {}
    tracker.base_threshold_tag = base_threshold_tag

    print(f"\n{'='*70}")
    print("LOADING MULTITHRESHOLD DATA (BASE LAYER)")
    print(f"{'='*70}")

    for om_id, (crown_file, ortho_file) in zip(tracker.om_ids, tracker.file_pairs):
        layers = _list_threshold_layers(crown_file)
        if not layers:
            raise RuntimeError(f"No layers found in {crown_file}")
        if base_threshold_tag is None:
            base_layer = max(layers, key=_threshold_tag_to_float)
        else:
            if base_threshold_tag not in layers:
                raise ValueError(f"Base threshold {base_threshold_tag} not found in {crown_file}. Available: {layers}")
            base_layer = base_threshold_tag
        tracker.multithreshold_layers[om_id] = layers
        tracker.base_threshold_tag = base_layer
        print(f"\nOM{om_id}: Loading {os.path.basename(crown_file)} layer {base_layer}")
        gdf = _load_layer_gdf(crown_file, base_layer)
        tracker.crowns_gdfs[om_id] = gdf
        tracker.crown_crs[om_id] = gdf.crs

        if load_images and ortho_file and os.path.exists(ortho_file):
            print(f"  Extracting {len(gdf)} crown images from {os.path.basename(ortho_file)}...")
            with rasterio.open(ortho_file) as src:
                tracker.ortho_crs[om_id] = src.crs
                patches: List[Optional[np.ndarray]] = []
                for _, row in gdf.iterrows():
                    geom = [mapping(row.geometry)]
                    try:
                        out_image, _ = mask(src, geom, crop=True)
                        img_patch = np.moveaxis(out_image, 0, -1)
                    except Exception:
                        img_patch = None
                    patches.append(img_patch)
            tracker.crown_images[om_id] = patches
            print(f"  ✓ Extracted {sum(1 for p in patches if p is not None)} images")
        else:
            tracker.crown_images[om_id] = [None] * len(gdf)
            if ortho_file and os.path.exists(ortho_file):
                with rasterio.open(ortho_file) as src:
                    tracker.ortho_crs[om_id] = src.crs
            else:
                tracker.ortho_crs[om_id] = None

        tracker.crown_attrs[om_id] = [
            tracker._compute_crown_attributes(row.geometry)
            for _, row in gdf.iterrows()
        ]

    tracker._check_crs_consistency()
    print(f"\n{'='*70}\n")

def build_multithreshold_cache(
    tracker: 'TreeTrackingGraph',
    include_base: bool = False,
    min_threshold_tag: Optional[str] = None
):
    cache: Dict[int, Dict[str, gpd.GeoDataFrame]] = {}
    all_thresholds = set()
    base_tag = tracker.base_threshold_tag
    base_val = _threshold_tag_to_float(base_tag) if base_tag else None
    min_val = _threshold_tag_to_float(min_threshold_tag) if min_threshold_tag else None

    for om_id, (crown_file, _) in zip(tracker.om_ids, tracker.file_pairs):
        layers = tracker.multithreshold_layers.get(om_id) or _list_threshold_layers(crown_file)
        filtered = []
        for layer in layers:
            if not include_base and base_tag and layer == base_tag:
                continue
            val = _threshold_tag_to_float(layer)
            if min_val is not None and (np.isnan(val) or val < min_val):
                continue
            if base_val is not None and not include_base and val > base_val:
                continue
            filtered.append(layer)
        cache[om_id] = {}
        for layer in filtered:
            gdf = _load_layer_gdf(crown_file, layer)
            gdf_aligned = _align_gdf_to_tracker(gdf, om_id, tracker)
            cache[om_id][layer] = gdf_aligned
            all_thresholds.add(layer)
    thresholds_desc = sorted(all_thresholds, key=_threshold_tag_to_float, reverse=True)
    return cache, thresholds_desc

def _append_crown_to_tracker(
    tracker: 'TreeTrackingGraph',
    om_id: int,
    row: Dict[str, Any],
    threshold_tag: str
) -> Tuple[int, int]:
    gdf = tracker.crowns_gdfs[om_id]
    new_row = dict(row)
    new_row['threshold_tag'] = threshold_tag
    new_row['is_augmented'] = True
    if 'confidence' not in new_row:
        new_row['confidence'] = np.nan
    gdf_new = pd.concat([gdf, gpd.GeoDataFrame([new_row], crs=gdf.crs)], ignore_index=True)
    tracker.crowns_gdfs[om_id] = gdf_new
    new_idx = len(gdf_new) - 1
    attrs = tracker._compute_crown_attributes(new_row['geometry'])
    tracker.crown_attrs[om_id].append(attrs)
    tracker.G.add_node((om_id, new_idx), **attrs)
    return (om_id, new_idx)

def _add_gap_edge(tracker: 'TreeTrackingGraph', src: Tuple[int, int], dst: Tuple[int, int], max_dist: float):
    src_attrs = tracker.crown_attrs[src[0]][src[1]]
    dst_attrs = tracker.crown_attrs[dst[0]][dst[1]]
    features = tracker._compute_pair_metrics(src_attrs, dst_attrs, max_dist=max_dist)
    base_similarity, parts = tracker._weighted_similarity(src_attrs, dst_attrs, max_dist=max_dist)
    tracker.G.add_edge(
        src, dst,
        similarity=float(base_similarity),
        method='gap_fill',
        case='gap_fill',
        overlap_prev=float(features['overlap_prev']),
        overlap_curr=float(features['overlap_curr']),
        iou=float(features['iou']),
        centroid_distance=float(features['centroid_dist']),
        base_similarity=float(base_similarity),
        spatial_similarity=float(parts['spatial']),
        area_similarity=float(parts['area']),
        shape_similarity=float(parts['shape'])
    )

def _best_gap_candidate(
    tracker: 'TreeTrackingGraph',
    om_id: int,
    prev_node: Optional[Tuple[int, int]],
    next_node: Optional[Tuple[int, int]],
    mt_cache: Dict[int, Dict[str, gpd.GeoDataFrame]],
    thresholds_desc: List[str],
    max_centroid_dist: float,
    min_iou: float,
    duplicate_iou: float
):
    prev_geom = tracker.crown_attrs[prev_node[0]][prev_node[1]]['geometry'] if prev_node else None
    next_geom = tracker.crown_attrs[next_node[0]][next_node[1]]['geometry'] if next_node else None
    existing_geoms = list(tracker.crowns_gdfs[om_id].geometry)
    best = None
    best_score = -1e9
    best_tag = None

    for tag in thresholds_desc:
        gdf = mt_cache.get(om_id, {}).get(tag)
        if gdf is None or gdf.empty:
            continue
        for _, row in gdf.iterrows():
            geom = row.geometry
            if geom is None or geom.is_empty:
                continue
            if any(tracker._compute_iou(geom, eg) >= duplicate_iou for eg in existing_geoms):
                continue
            scores = []
            if prev_geom is not None:
                dist_prev = geom.centroid.distance(prev_geom.centroid)
                if dist_prev > max_centroid_dist:
                    continue
                iou_prev = tracker._compute_iou(geom, prev_geom)
                if iou_prev < min_iou:
                    continue
                scores.append((iou_prev, dist_prev))
            if next_geom is not None:
                dist_next = geom.centroid.distance(next_geom.centroid)
                if dist_next > max_centroid_dist:
                    continue
                iou_next = tracker._compute_iou(geom, next_geom)
                if iou_next < min_iou:
                    continue
                scores.append((iou_next, dist_next))
            if not scores:
                continue
            score = sum(iou for iou, _ in scores) - (sum(dist for _, dist in scores) / max_centroid_dist)
            if score > best_score:
                best_score = score
                best = row
                best_tag = tag
        if best is not None:
            break
    return best, best_tag

def augment_partial_chains_with_multithreshold(
    tracker: 'TreeTrackingGraph',
    base_chain_categories: Dict[str, List[List[Tuple[int, int]]]],
    mt_cache: Dict[int, Dict[str, gpd.GeoDataFrame]],
    thresholds_desc: List[str],
    max_centroid_dist: float = 25.0,
    min_iou: float = 0.20,
    duplicate_iou: float = 0.70
):
    augmented = 0
    augmented_by_threshold = defaultdict(int)
    chains_to_extend = base_chain_categories.get('partial_long', []) + base_chain_categories.get('partial_short', [])
    om_ids = tracker.om_ids
    min_om = min(om_ids)
    max_om = max(om_ids)

    for chain in chains_to_extend:
        while True:
            last_node = chain[-1]
            last_om = last_node[0]
            if last_om >= max_om:
                break
            if tracker.G.out_degree(last_node) > 0:
                break
            target_om = last_om + 1
            if target_om not in om_ids:
                break
            cand, tag = _best_gap_candidate(
                tracker, target_om, last_node, None, mt_cache, thresholds_desc,
                max_centroid_dist=max_centroid_dist, min_iou=min_iou, duplicate_iou=duplicate_iou
            )
            if cand is None:
                break
            new_node = _append_crown_to_tracker(tracker, target_om, cand.to_dict(), tag)
            _add_gap_edge(tracker, last_node, new_node, max_dist=max_centroid_dist)
            chain.append(new_node)
            augmented += 1
            augmented_by_threshold[tag] += 1
        while True:
            first_node = chain[0]
            first_om = first_node[0]
            if first_om <= min_om:
                break
            if tracker.G.in_degree(first_node) > 0:
                break
            target_om = first_om - 1
            if target_om not in om_ids:
                break
            cand, tag = _best_gap_candidate(
                tracker, target_om, None, first_node, mt_cache, thresholds_desc,
                max_centroid_dist=max_centroid_dist, min_iou=min_iou, duplicate_iou=duplicate_iou
            )
            if cand is None:
                break
            new_node = _append_crown_to_tracker(tracker, target_om, cand.to_dict(), tag)
            _add_gap_edge(tracker, new_node, first_node, max_dist=max_centroid_dist)
            chain.insert(0, new_node)
            augmented += 1
            augmented_by_threshold[tag] += 1

    summary = {
        'augmented_nodes': augmented,
        'by_threshold': dict(augmented_by_threshold),
    }
    return summary

print("✓ Multithreshold helpers loaded (PCC-only alignment enforced)")

In [ ]:
# Patch layer listing: validate meta thresholds against actual GPKG layers
def _list_threshold_layers(gpkg_path: str) -> List[str]:
    meta_path = os.path.splitext(gpkg_path)[0] + ".meta.json"
    meta_layers = []
    if os.path.exists(meta_path):
        try:
            with open(meta_path, "r") as f:
                meta = json.load(f)
            raw_thresholds = meta.get("thresholds", [])
            for t in raw_thresholds:
                if isinstance(t, str) and t.startswith("conf_"):
                    meta_layers.append(t)
                elif isinstance(t, (int, float)):
                    meta_layers.append(_float_to_threshold_tag(float(t)))
            meta_layers = list(dict.fromkeys(meta_layers))
        except Exception as e:
            print(f"⚠️ Failed reading meta: {meta_path} ({e})")
    actual_layers = []
    if FIONA_AVAILABLE:
        try:
            actual_layers = fiona.listlayers(gpkg_path)
        except Exception as e:
            print(f"⚠️ Failed listing layers in {gpkg_path} ({e})")
    if actual_layers:
        if meta_layers:
            layers = [l for l in meta_layers if l in actual_layers]
        else:
            layers = actual_layers
    else:
        layers = meta_layers
    if not layers:
        raise RuntimeError(f"No layers found in {gpkg_path}")
    return sorted(layers, key=_threshold_tag_to_float)

print("✓ Patched layer listing to validate against actual GPKG layers")

## 3. Visualization Functions

Functions for visualizing tracking chains with polygons and extracted RGB images.

In [ ]:
def visualize_chain_with_extracted_images(chain, aligned_gdfs, crown_images_dict, tracker, title="Chain Example", save_path=None, show=True, close_fig=True, dpi=150):
    """
    Visualize chain with both crown polygons AND extracted RGB images.
    Optional saving for PPT-friendly reuse.

    Args:
        chain: List of (om_id, crown_idx) tuples
        aligned_gdfs: Dictionary of aligned GeoDataFrames by OM ID
        crown_images_dict: Dictionary of crown images by OM ID (from tracker.crown_images)
        tracker: TreeTrackingGraph instance
        title: Plot title
        save_path: Optional path to save the figure
        show: Whether to display the plot
        close_fig: Close the figure after saving/showing to free memory
        dpi: Resolution when saving
    """
    chain_length = len(chain)
    
    # Create figure with 2 rows: polygons on top, images on bottom
    fig = plt.figure(figsize=(5*chain_length, 10))
    
    print(f"\n{'='*80}")
    print(f"{title}")
    print(f"{'='*80}")
    print(f"Chain: {chain}")
    print(f"Length: {chain_length}\n")
    
    # Plot each crown in the chain
    for idx, (om_idx, crown_idx) in enumerate(chain):
        gdf = aligned_gdfs[om_idx]
        crown = gdf.iloc[crown_idx]
        
        # Get edge info if not last crown
        edge_info = ""
        if idx < chain_length - 1:
            next_node = chain[idx + 1]
            if tracker.G.has_edge((om_idx, crown_idx), next_node):
                edge_data = tracker.G.edges[(om_idx, crown_idx), next_node]
                edge_info = f" → OM{next_node[0]}[{next_node[1]}]: case={edge_data['case']}, sim={edge_data['similarity']:.2f}, iou={edge_data.get('iou', 0):.2f}"
        
        # Print crown info
        in_deg = tracker.G.in_degree((om_idx, crown_idx))
        out_deg = tracker.G.out_degree((om_idx, crown_idx))
        print(f"  OM{om_idx}[{crown_idx}]: in_deg={in_deg}, out_deg={out_deg}, area={crown.geometry.area:.1f}m²{edge_info}")
        
        # ROW 1: Crown polygon
        ax_poly = plt.subplot(2, chain_length, idx + 1)
        
        # Get bounds and plot
        minx, miny, maxx, maxy = crown.geometry.bounds
        margin = max((maxx - minx), (maxy - miny)) * 0.3
        
        # Plot other crowns in gray
        gdf.plot(ax=ax_poly, color='lightgray', edgecolor='gray', alpha=0.3)
        
        # Plot this crown highlighted
        gpd.GeoSeries([crown.geometry]).plot(
            ax=ax_poly,
            facecolor=plt.cm.tab10(om_idx-1),
            edgecolor='black',
            linewidth=2,
            alpha=0.7
        )
        
        # Plot centroid
        centroid = crown.geometry.centroid
        ax_poly.plot(centroid.x, centroid.y, 'o', color='yellow', markersize=8,
                    markeredgecolor='black', markeredgewidth=1.5)
        
        # Arrow to next crown
        if idx < chain_length - 1:
            ax_poly.annotate('', xy=(maxx + margin*0.5, (miny+maxy)/2),
                            xytext=(maxx + margin*0.2, (miny+maxy)/2),
                            arrowprops=dict(arrowstyle='->', lw=3, color='red'))
        
        ax_poly.set_xlim(minx - margin, maxx + margin)
        ax_poly.set_ylim(miny - margin, maxy + margin)
        ax_poly.set_aspect('equal')
        ax_poly.set_title(
            f"OM{om_idx} - Crown {crown_idx}\nArea: {crown.geometry.area:.1f}m²\nIn:{in_deg} Out:{out_deg}",
            fontsize=10, fontweight='bold'
        )
        ax_poly.set_xlabel("X (m)", fontsize=9)
        ax_poly.set_ylabel("Y (m)", fontsize=9)
        ax_poly.grid(True, alpha=0.3)
        
        # Add edge case label
        if edge_info:
            edge_data = tracker.G.edges[(om_idx, crown_idx), chain[idx + 1]]
            bbox_color = {
                'one_to_one': 'wheat',
                'containment': 'lightblue',
                'nearby': 'lightcoral'
            }.get(edge_data['case'], 'white')
            ax_poly.text(
                0.95, 0.95,
                f"→ {edge_data['case']}\nSim: {edge_data['similarity']:.2f}\nIoU: {edge_data.get('iou', 0):.2f}",
                transform=ax_poly.transAxes, fontsize=8,
                verticalalignment='top', horizontalalignment='right',
                bbox=dict(boxstyle='round', facecolor=bbox_color, alpha=0.8)
            )
        
        # ROW 2: Extracted RGB image
        ax_img = plt.subplot(2, chain_length, chain_length + idx + 1)
        
        rgb_image = crown_images_dict[om_idx][crown_idx]
        if rgb_image is not None:
            ax_img.imshow(rgb_image)
            ax_img.set_title(
                f"Extracted Tree Image\n{rgb_image.shape[0]}×{rgb_image.shape[1]} px",
                fontsize=9
            )
        else:
            ax_img.text(0.5, 0.5, 'Image not available',
                       ha='center', va='center', fontsize=10, color='red')
            ax_img.set_title("No Image", fontsize=9)
        
        ax_img.axis('off')
    
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=dpi, bbox_inches='tight')
    if show:
        plt.show()
    if close_fig:
        plt.close(fig)
    print()
    return save_path


def categorize_chains(tracker):
    """
    Categorize all chains from the tracker.
    
    Returns:
        Dictionary with categorized chains
    """
    all_chains = tracker._extract_all_chains()
    max_chain_length = len(tracker.om_ids)
    
    full_chains_width_1 = []      # Full length chains with no branching
    full_chains_branching = []    # Full length chains with branching
    partial_chains_long = []      # Length 3-4
    partial_chains_short = []     # Length 2
    
    for chain in all_chains:
        chain_length = len(chain)
        
        if chain_length == max_chain_length:
            # Full chain - check for branching
            has_branching = False
            for node in chain:
                in_degree = tracker.G.in_degree(node)
                out_degree = tracker.G.out_degree(node)
                if in_degree > 1 or out_degree > 1:
                    has_branching = True
                    break
            
            if has_branching:
                full_chains_branching.append(chain)
            else:
                full_chains_width_1.append(chain)
        else:
            # Partial chains
            if chain_length >= 3:
                partial_chains_long.append(chain)
            elif chain_length == 2:
                partial_chains_short.append(chain)
    
    return {
        'full_width_1': full_chains_width_1,
        'full_branching': full_chains_branching,
        'partial_long': partial_chains_long,
        'partial_short': partial_chains_short,
        'singleton': [c for c in all_chains if len(c) == 1]
    }


print("✓ Visualization functions loaded")

In [ ]:
# Enhanced TreeTrackingGraph that supports multi-threshold crowns
# Key modifications:
# 1. Can load specific layers from multi-threshold GPKGs
# 2. Uses phase cross-correlation alignment ONLY  
# 3. More relaxed matching parameters

import pandas as pd

class MultiThresholdTreeTracker(TreeTrackingGraph):
    """
    Extended tracker that can use multi-threshold crown detections.
    
    Strategy:
    - OM1: Use 0.40 threshold as reference (balanced quality/quantity)
    - OM2-7: Use selected thresholds combined, then filter best matches
    """
    
    def __init__(self, multithresh_dir: str, ortho_dir: str, output_dir: str = '../../output', **kwargs):
        # Don't call super().__init__ because we'll handle discovery ourselves
        self.multithresh_dir = Path(multithresh_dir)
        self.ortho_dir = ortho_dir or self._resolve_dir('input/input_om', '../../input/input_om')
        self.output_dir = output_dir
        self.simplify_tol = kwargs.get('simplify_tol', 1.0)
        self.max_crowns_preview = kwargs.get('max_crowns_preview', 200)
        
        # Storage
        self.crowns_gdfs: Dict[int, gpd.GeoDataFrame] = {}
        self.crown_attrs: Dict[int, List[Dict[str, Any]]] = {}
        self.crown_images: Dict[int, List[Optional[np.ndarray]]] = {}
        self.crown_crs: Dict[int, Optional[Any]] = {}
        self.ortho_crs: Dict[int, Optional[Any]] = {}
        self.G: nx.DiGraph = nx.DiGraph()
        self.match_case_mode: str = "balanced"
        
        # Multi-threshold specific
        self.om_threshold_layers: Dict[int, str] = {}  # om_id -> layer name used
        
        self.case_configs: Dict[str, MatchCaseConfig] = self._ultra_relaxed_case_configs()
        self.case_order: List[str] = ['one_to_one', 'containment', 'nearby']
        self.last_case_counts: Dict[str, int] = {}
        self.last_selected_counts: Dict[str, int] = {}
        
        # Discover files
        self.discover_multithreshold_files()
    
    def discover_multithreshold_files(self):
        """Discover multi-threshold GPKG files and pair with orthomosaics."""
        gpkg_files = sorted(self.multithresh_dir.glob('OM*_multithreshold.gpkg'))
        ortho_files = sorted(Path(self.ortho_dir).glob('sit_om*.tif')) if os.path.exists(self.ortho_dir) else []
        
        print(f"Found {len(gpkg_files)} multi-threshold GPKG files")
        print(f"Found {len(ortho_files)} orthomosaic files")
        
        # Extract OM IDs
        self.om_ids = []
        self.file_pairs = []
        
        for gpkg in gpkg_files:
            # Extract OM number from filename like "OM1_multithreshold.gpkg"
            om_num = int(gpkg.stem.split('_')[0].replace('OM', ''))
            self.om_ids.append(om_num)
            
            # Find matching orthomosaic
            ortho_match = None
            for ortho in ortho_files:
                if f'om{om_num}' in ortho.stem.lower():
                    ortho_match = str(ortho)
                    break
            
            self.file_pairs.append((str(gpkg), ortho_match))
        
        print(f"Paired {len(self.file_pairs)} OM files (IDs: {self.om_ids})")
    
    def load_multithreshold_data(
        self,
        om1_threshold: str = 'conf_0p4',
        other_thresholds: List[str] = None,
        load_images: bool = False,
        align: bool = True
    ):
        """
        Load crown data using multi-threshold strategy.
        
        Args:
            om1_threshold: Layer to use for OM1 reference (e.g., 'conf_0p4')
            other_thresholds: Layers to try for other OMs (default: selected thresholds)
            load_images: Extract RGB patches
            align: Use phase cross-correlation alignment
        """
        if other_thresholds is None:
            other_thresholds = ['conf_0p35', 'conf_0p4', 'conf_0p45']
        
        print(f"\n{'='*70}")
        print("LOADING MULTI-THRESHOLD CROWN DATA")
        print(f"{'='*70}")
        print(f"OM1 reference threshold: {om1_threshold}")
        print(f"Other OMs thresholds: {other_thresholds}")
        print()
        
        for om_id, (crown_file, ortho_file) in zip(self.om_ids, self.file_pairs):
            # OM1: Use specified threshold
            if om_id == 1:
                layer = om1_threshold
                gdf = gpd.read_file(crown_file, layer=layer)
                print(f"OM{om_id}: Loaded {len(gdf)} crowns from layer '{layer}'")
                self.om_threshold_layers[om_id] = layer
            else:
                # Other OMs: Combine all threshold layers
                gdfs = []
                for layer in other_thresholds:
                    try:
                        layer_gdf = gpd.read_file(crown_file, layer=layer)
                        gdfs.append(layer_gdf)
                    except Exception as e:
                        print(f"  Warning: Could not load layer {layer} from OM{om_id}: {e}")
                
                if not gdfs:
                    print(f"  ERROR: No layers loaded for OM{om_id}, skipping")
                    continue
                
                # Concatenate all layers
                gdf = pd.concat(gdfs, ignore_index=True)
                
                # Remove duplicates based on geometry (approximate)
                # Use a buffer of 0 to normalize geometries, then drop exact duplicates
                gdf['geom_wkt'] = gdf.geometry.apply(lambda g: g.buffer(0).wkt)
                gdf = gdf.drop_duplicates(subset='geom_wkt').drop(columns='geom_wkt')
                
                # CRITICAL: Reset index after deduplication to ensure crown_attrs alignment
                gdf = gdf.reset_index(drop=True)
                
                print(f"OM{om_id}: Loaded {len(gdf)} unique crowns from {len(gdfs)} threshold layers")
                self.om_threshold_layers[om_id] = f"combined_{len(other_thresholds)}_thresholds"
            
            # Reset index for OM1 too to be consistent
            if om_id == 1:
                gdf = gdf.reset_index(drop=True)
            
            self.crowns_gdfs[om_id] = gdf
            self.crown_crs[om_id] = gdf.crs
            
            # Extract images from ORIGINAL geometries BEFORE alignment
            if load_images and ortho_file and os.path.exists(ortho_file):
                print(f"  Extracting {len(gdf)} crown images from {os.path.basename(ortho_file)}...")
                with rasterio.open(ortho_file) as src:
                    self.ortho_crs[om_id] = src.crs
                    patches: List[Optional[np.ndarray]] = []
                    for _, row in gdf.iterrows():
                        geom = [mapping(row.geometry)]
                        try:
                            out_image, _ = mask(src, geom, crop=True)
                            img_patch = np.moveaxis(out_image, 0, -1)
                        except Exception:
                            img_patch = None
                        patches.append(img_patch)
                self.crown_images[om_id] = patches
                print(f"  ✓ Extracted {sum(1 for p in patches if p is not None)} images")
            else:
                self.crown_images[om_id] = [None] * len(gdf)
                if ortho_file and os.path.exists(ortho_file):
                    with rasterio.open(ortho_file) as src:
                        self.ortho_crs[om_id] = src.crs
            
            # Compute attributes
            self.crown_attrs[om_id] = [
                self._compute_crown_attributes(row.geometry) 
                for _, row in gdf.iterrows()
            ]
        
        self._check_crs_consistency()
        print(f"\n{'='*70}")
        
        # Apply phase cross-correlation alignment
        if align:
            print(f"\n{'='*70}")
            print("APPLYING PHASE CROSS-CORRELATION ALIGNMENT")
            print(f"{'='*70}")
            
            if not SKIMAGE_AVAILABLE:
                print("WARNING: scikit-image not available, skipping alignment")
            else:
                self.alignment_shifts = phase_corr_align_tracker(self, reference_om_id=1)
                print(f"{'='*70}\n")

print("✓ MultiThresholdTreeTracker class defined")

---

## 🎯 Multi-Threshold Crown Tracking: High-Quality Approach

This section implements crown tracking across all 7 orthomosaics using multi-threshold crown detections with **very strict matching parameters** to ensure only high-quality matches.

**Strategy:**
- OM1: Use threshold 0.45 (high-quality base, ~96 crowns)
- OM2-7: Use thresholds 0.15-0.45 (wide range for maximum candidates)
- **Very strict matching after PCC alignment:** Max distances 10-20m, min IoU 30%
- Extract backbones from branching chains using intelligent edge selection

**Matching Parameters (Post-PCC Alignment):**
- **Weight Priority:** IoU (45%) > Spatial (35%) > Area (15%) > Shape (5%)
- **one_to_one**: max_dist=**10m**, min_iou=**30%**, min_overlap=20%, min_similarity=40%
- **containment**: max_dist=**15m**, min_iou=**30%**, min_overlap=30%, min_similarity=35%
- **nearby**: max_dist=**20m**, min_iou=10%, min_overlap=10%, min_similarity=30%

**Edge Selection Priority:**
1. one_to_one edges (highest confidence)
2. containment edges (medium confidence)
3. nearby edges (spatial-based, lower confidence)

In [ ]:
# Explore available multi-threshold data
multithresh_dir = Path('../../output/input_crowns_multithreshold_min0p15')

meta_files = sorted(multithresh_dir.glob('*.meta.json'))
print(f"Found {len(meta_files)} multi-threshold metadata files\n")

# Display threshold coverage
for meta_file in meta_files[:1]:  # Show OM1 as example
    with open(meta_file) as f:
        meta = json.load(f)
    print(f"{meta['orthomosaic']}: {meta['thresholds']}")
    print(f"  Layers: {list(meta['layer_counts'].keys())}\n")

In [ ]:
# Initialize multi-threshold tracker
tracker_final = MultiThresholdTreeTracker(
    multithresh_dir='../../output/input_crowns_multithreshold_min0p15',
    ortho_dir='../../input/input_om',
    output_dir='../../output'
)

# Load with high-quality base + wide threshold range  
tracker_final.load_multithreshold_data(
    om1_threshold='conf_0p45',  # HIGH QUALITY base (~96 crowns)
    other_thresholds=['conf_0p15', 'conf_0p2', 'conf_0p25', 'conf_0p3', 'conf_0p35', 'conf_0p4', 'conf_0p45'],
    load_images=True,
    align=True
)

print(f"\nLoaded crowns:")
for om_id in tracker_final.om_ids:
    n_crowns = len(tracker_final.crowns_gdfs[om_id])
    print(f"  OM{om_id}: {n_crowns} crowns")
print(f"\nTotal: {sum(len(gdf) for gdf in tracker_final.crowns_gdfs.values())} crowns")

In [ ]:
# Define very strict matching config for well-aligned orthomosaics (post-PCC)
def make_strict_aligned_configs():
    """
    Very conservative matching parameters for well-aligned orthomosaics.
    After phase cross-correlation, we expect tight spatial correspondence.
    
    Weight priority: IoU > Spatial > Area > Shape
    Very low centroid distances (5-10m) to avoid false matches.
    """
    return {
        'one_to_one': MatchCaseConfig(
            name='one_to_one',
            base_similarity_weights={'iou': 0.45, 'spatial': 0.35, 'area': 0.15, 'shape': 0.05},
            scoring_weights={'base': 0.40, 'iou': 0.30, 'overlap_prev': 0.10, 'overlap_curr': 0.10, 'centroid': 0.10},
            similarity_threshold=0.40, min_iou=0.30, min_overlap_prev=0.20, min_overlap_curr=0.20,
            max_centroid_dist=10.0, mutual_best=False, allow_multiple=True,
            max_edges_per_prev=2, max_edges_per_curr=2
        ),
        'containment': MatchCaseConfig(
            name='containment',
            base_similarity_weights={'iou': 0.40, 'spatial': 0.35, 'area': 0.20, 'shape': 0.05},
            scoring_weights={'base': 0.30, 'overlap_prev': 0.35, 'overlap_curr': 0.35},
            similarity_threshold=0.35, min_iou=0.30, min_overlap_prev=0.30, min_overlap_curr=0.30,
            max_centroid_dist=15.0, mutual_best=False, allow_multiple=True,
            max_edges_per_prev=2, max_edges_per_curr=2
        ),
        'nearby': MatchCaseConfig(
            name='nearby',
            base_similarity_weights={'iou': 0.35, 'spatial': 0.45, 'area': 0.15, 'shape': 0.05},
            scoring_weights={'base': 0.60, 'centroid': 0.40},
            similarity_threshold=0.30, min_iou=0.10, min_overlap_prev=0.10, min_overlap_curr=0.10,
            max_centroid_dist=20.0, mutual_best=False, allow_multiple=True,
            max_edges_per_prev=2, max_edges_per_curr=2
        ),
    }

print("✓ Very strict aligned config defined (IoU-prioritized, max_dist=10/15/20m, min_iou=0.30)")

In [ ]:
# Build tracking graph with very strict parameters
tracker_final.case_configs = make_strict_aligned_configs()

tracker_final.build_graph_conditional(
    base_max_dist=30.0,    # Very tight search - well-aligned orthomosaics
    overlap_gate=0.10,     # Higher overlap requirement - 10% minimum
    min_base_similarity=0.30,  # Only consider strong matches
    max_candidates_per_prev=None,
    max_candidates_per_curr=None,
    classify_mode='balanced'
)

# Generate quality report
quality_text, quality_metrics = tracker_final.quality_report()
print(quality_text)

tracker_final.save_text(quality_text, 'final_tracking_quality_report.txt')
tracker_final.save_json(quality_metrics, 'final_tracking_quality_metrics.json')

In [ ]:
# Categorize chains: clean vs branching
chains_all = tracker_final._extract_all_chains()
categories = categorize_chains(tracker_final)

clean_chains = categories['full_width_1']
branching_chains = categories['full_branching']

print(f"Full-length chains (7 OMs): {len(clean_chains) + len(branching_chains)}")
print(f"  Non-branching (clean): {len(clean_chains)}")
print(f"  With branching: {len(branching_chains)}")

In [ ]:
# Extract backbones from branching chains using mixed-edge strategy
# Preference: one_to_one > containment > nearby
# Conflict resolution: max similarity

def extract_backbone_mixed(tracker, chain_nodes):
    """
    Extract best path through chain allowing mixed edge types.
    
    Args:
        tracker: The tracking graph object
        chain_nodes: List of (om_id, crown_idx) tuples forming a chain
    
    Returns:
        List of edge dictionaries forming the backbone, or None if extraction fails
    """
    # Get all edges between consecutive nodes in the chain
    edges_by_step = {}
    
    for i in range(len(chain_nodes) - 1):
        node1 = chain_nodes[i]
        node2 = chain_nodes[i + 1]
        
        # Find all edges from node1 to node2 in the graph
        step_edges = []
        if tracker.G.has_edge(node1, node2):
            edge_data = tracker.G.edges[node1, node2]
            step_edges.append({
                'node1': node1,
                'node2': node2,
                'match_type': edge_data['case'],
                'similarity': edge_data['similarity'],
                'iou': edge_data.get('iou', 0),
                'om_pair': f"OM{node1[0]}_OM{node2[0]}"
            })
        
        # Also check for any alternative edges at this step (branching)
        for successor in tracker.G.successors(node1):
            if successor != node2 and successor[0] == node2[0]:  # Same OM, different crown
                edge_data = tracker.G.edges[node1, successor]
                step_edges.append({
                    'node1': node1,
                    'node2': successor,
                    'match_type': edge_data['case'],
                    'similarity': edge_data['similarity'],
                    'iou': edge_data.get('iou', 0),
                    'om_pair': f"OM{node1[0]}_OM{successor[0]}"
                })
        
        if not step_edges:
            return None
        
        edges_by_step[i] = step_edges
    
    # Quality hierarchy
    quality_map = {'one_to_one': 3, 'containment': 2, 'nearby': 1}
    
    # Build path by selecting best edge at each step
    path = []
    
    for i in range(len(chain_nodes) - 1):
        candidates = edges_by_step[i]
        
        # Select best edge: (quality, similarity)
        best = max(candidates, key=lambda e: (quality_map.get(e['match_type'], 0), e['similarity']))
        path.append(best)
    
    return path

extracted_backbones = []
for chain in branching_chains:
    backbone = extract_backbone_mixed(tracker_final, chain)
    if backbone:
        # chain is a list of nodes, backbone is list of edge dicts
        extracted_backbones.append({
            'edges': backbone,
            'crowns': [chain[0]] + [e['node2'] for e in backbone],
            'avg_similarity': np.mean([e['similarity'] for e in backbone]),
            'edge_types': [e['match_type'] for e in backbone],
            'one_to_one_count': sum(1 for e in backbone if e['match_type'] == 'one_to_one')
        })

print(f"Extracted {len(extracted_backbones)} backbones from {len(branching_chains)} branching chains")
pure_one_to_one = [b for b in extracted_backbones if b['one_to_one_count'] == 6]
mixed_edges = [b for b in extracted_backbones if b['one_to_one_count'] < 6]
print(f"  Pure one_to_one paths: {len(pure_one_to_one)}")
print(f"  Mixed-edge paths: {len(mixed_edges)}")

In [ ]:
# Combine all extracted chains (clean + backbones)
all_extracted_chains = clean_chains + extracted_backbones

print(f"\n{'='*60}")
print(f"FINAL RESULT: {len(all_extracted_chains)} high-quality tracking chains")
print(f"{'='*60}")
print(f"  Clean (non-branching): {len(clean_chains)}")
print(f"  Extracted backbones: {len(extracted_backbones)}")
print(f"    - Pure one_to_one: {len(pure_one_to_one)}")
print(f"    - Mixed edges: {len(mixed_edges)}")

# Quality summary
if extracted_backbones:
    avg_sim_pure = np.mean([b['avg_similarity'] for b in pure_one_to_one]) if pure_one_to_one else 0
    avg_sim_mixed = np.mean([b['avg_similarity'] for b in mixed_edges]) if mixed_edges else 0
    print(f"\nQuality metrics:")
    print(f"  Pure one_to_one avg similarity: {avg_sim_pure:.3f}")
    print(f"  Mixed-edge avg similarity: {avg_sim_mixed:.3f}")
    if mixed_edges:
        avg_one_to_one_pct = np.mean([b['one_to_one_count'] / 6 * 100 for b in mixed_edges])
        print(f"  Mixed-edge avg one_to_one %: {avg_one_to_one_pct:.1f}%")

In [ ]:
# Visualize all tracked chains with tree images
import os
os.makedirs('../../output/tracked_chains_visualization', exist_ok=True)

print(f"Visualizing {len(all_extracted_chains)} tracked chains...")
print(f"{'='*70}\n")

for idx, chain_data in enumerate(all_extracted_chains, 1):
    # Extract chain (list of nodes)
    if 'edges' in chain_data:
        # Extracted backbone - reconstruct chain from edges
        chain = [chain_data['edges'][0]['node1']] + [e['node2'] for e in chain_data['edges']]
    else:
        # Clean chain - already a list
        chain = chain_data
    
    # Get first crown coordinates
    om_id, crown_idx = chain[0]
    first_crown = tracker_final.crowns_gdfs[om_id].iloc[crown_idx]
    centroid = first_crown.geometry.centroid
    coords = f"({centroid.x:.1f}, {centroid.y:.1f})"
    
    # Determine quality
    if 'one_to_one_count' in chain_data:
        if chain_data['one_to_one_count'] == 6:
            quality = "Excellent (100% one_to_one)"
        else:
            oto_pct = chain_data['one_to_one_count'] / 6 * 100
            quality = f"Good ({oto_pct:.0f}% one_to_one)"
    else:
        quality = "Excellent (Clean, no branching)"
    
    title = f"Chain {idx}/{len(all_extracted_chains)} - {quality}"
    save_path = f"../../output/tracked_chains_visualization/chain_{idx:02d}.png"
    
    print(f"Chain {idx}: {coords} - {quality}")
    
    # Visualize with images
    visualize_chain_with_extracted_images(
        chain=chain,
        aligned_gdfs=tracker_final.crowns_gdfs,
        crown_images_dict=tracker_final.crown_images,
        tracker=tracker_final,
        title=title,
        save_path=save_path,
        show=False,  # Don't show interactively (too many)
        close_fig=True,
        dpi=150
    )

print(f"\n{'='*70}")
print(f"✓ Saved all {len(all_extracted_chains)} chain visualizations to:")
print(f"  output/tracked_chains_visualization/")
print(f"{'='*70}")

### 📊 Quality Improvements with Very Strict Matching

**Key Metrics:**
- **Total tracking chains:** 35 (high-quality only)
- **Clean chains (no branching):** 6 (17%)
- **Extracted backbones:** 29 (83%)
  - Pure one_to_one: 27/29 (93%)
  - Mixed edges: 2/29 (7%)
- **Average similarity:** 0.81 (excellent)
- **Edge quality:** 92% one_to_one edges (1123/1217 total edges)

**Strict Parameters Applied:**
- Max centroid distances: **10m** (one_to_one), **15m** (containment), **20m** (nearby)
- Min IoU requirement: **30%** (ensures strong spatial overlap)
- Weight priority: **IoU (45%) > Spatial (35%) > Area (15%) > Shape (5%)**
- Base search radius: **30m** only
- Min overlap gate: **10%**

**Benefits:**
✅ No false matches between distant crowns  
✅ Very high confidence in tracking (avg similarity 0.81)  
✅ 93% of backbones are pure one_to_one paths  
✅ Stricter matching = fewer chains but much higher quality  

In [ ]:
# Verification: Check that all visualizations match bad matches are filtered out
import os

viz_dir = '../../output/tracked_chains_visualization'
viz_files = sorted([f for f in os.listdir(viz_dir) if f.startswith('chain_') and f.endswith('.png')])

print(f"✓ Generated {len(viz_files)} high-quality chain visualizations")
print(f"✓ Location: {viz_dir}/")
print(f"\nVerification:")
print(f"  - Expected chains: {len(all_extracted_chains)}")
print(f"  - Visualization files: {len(viz_files)}")
print(f"  - Match: {'✓ YES' if len(viz_files) == len(all_extracted_chains) else '✗ NO'}")

# Sample a few chains to verify quality
print(f"\nSample chain quality (first 5):")
for i, chain_data in enumerate(all_extracted_chains[:5], 1):
    if 'edges' in chain_data:
        avg_sim = chain_data['avg_similarity']
        oto = chain_data['one_to_one_count']
        chain = [chain_data['edges'][0]['node1']] + [e['node2'] for e in chain_data['edges']]
    else:
        avg_sim = 1.0
        oto = 6
        chain = chain_data
    
    # Check centroid distances
    max_dist = 0
    for j in range(len(chain)-1):
        om1, idx1 = chain[j]
        om2, idx2 = chain[j+1]
        c1 = tracker_final.crowns_gdfs[om1].iloc[idx1].geometry.centroid
        c2 = tracker_final.crowns_gdfs[om2].iloc[idx2].geometry.centroid
        dist = c1.distance(c2)
        max_dist = max(max_dist, dist)
    
    print(f"  Chain {i}: sim={avg_sim:.3f}, one_to_one={oto}/6, max_dist={max_dist:.1f}m")

In [ ]:
# Analyze chain lengths and quality for length 5 and 6 chains
from collections import defaultdict

chain_stats = defaultdict(lambda: {'total': 0, 'clean': 0, 'pure_oto': 0, 'mixed': 0})

for chain_data in all_extracted_chains:
    # Determine chain length
    if 'edges' in chain_data:
        # Extracted backbone
        chain_length = len(chain_data['edges']) + 1  # edges + 1 = nodes
        oto_count = chain_data['one_to_one_count']
        
        if oto_count == 6:
            quality_type = 'pure_oto'
        else:
            quality_type = 'mixed'
    else:
        # Clean chain
        chain_length = len(chain_data)
        quality_type = 'clean'
    
    chain_stats[chain_length]['total'] += 1
    chain_stats[chain_length][quality_type] += 1

# Display results
print("Chain Analysis by Length:")
print("="*70)
for length in sorted(chain_stats.keys()):
    stats = chain_stats[length]
    print(f"\nLength {length}: {stats['total']} chains")
    print(f"  - Clean (no branching): {stats['clean']}")
    print(f"  - Pure one_to_one backbones: {stats['pure_oto']}")
    print(f"  - Mixed-edge backbones: {stats['mixed']}")
    
    good_oto = stats['clean'] + stats['pure_oto']
    print(f"  → Good one_to_one chains: {good_oto} ({good_oto/stats['total']*100:.1f}%)")

# Summary for length 5 and 6
print("\n" + "="*70)
print("ANSWER: Length 5 and 6 Good One-to-One Chains")
print("="*70)

if 5 in chain_stats:
    good_5 = chain_stats[5]['clean'] + chain_stats[5]['pure_oto']
    print(f"Length 5: {good_5} good one_to_one chains (out of {chain_stats[5]['total']} total)")
else:
    print("Length 5: 0 chains")

if 6 in chain_stats:
    good_6 = chain_stats[6]['clean'] + chain_stats[6]['pure_oto']
    print(f"Length 6: {good_6} good one_to_one chains (out of {chain_stats[6]['total']} total)")
else:
    print("Length 6: 0 chains")

total_good_5_6 = (chain_stats[5]['clean'] + chain_stats[5]['pure_oto'] if 5 in chain_stats else 0) + \
                 (chain_stats[6]['clean'] + chain_stats[6]['pure_oto'] if 6 in chain_stats else 0)
print(f"\nTotal length 5+6 good one_to_one chains: {total_good_5_6}")

In [ ]:
# Check all chains in the graph (including partial chains)
print("\n" + "="*70)
print("Checking ALL chains in the tracking graph (including partial):")
print("="*70)

all_graph_chains = tracker_final._extract_all_chains()
print(f"\nTotal chains in graph: {len(all_graph_chains)}")

# Categorize all chains
all_categories = categorize_chains(tracker_final)

# Analyze length 5 and 6 chains
length_5_chains = [c for c in all_graph_chains if len(c) == 5]
length_6_chains = [c for c in all_graph_chains if len(c) == 6]

print(f"\nLength 5 chains: {len(length_5_chains)}")
print(f"Length 6 chains: {len(length_6_chains)}")

# Check quality of these partial chains
def check_chain_quality(chain, tracker):
    """Check if a chain has all one_to_one edges."""
    if len(chain) < 2:
        return False, 0, 0
    
    total_edges = len(chain) - 1
    one_to_one_count = 0
    
    for i in range(len(chain) - 1):
        node1 = chain[i]
        node2 = chain[i + 1]
        
        if tracker.G.has_edge(node1, node2):
            edge_data = tracker.G.edges[node1, node2]
            if edge_data['case'] == 'one_to_one':
                one_to_one_count += 1
    
    is_all_oto = (one_to_one_count == total_edges)
    return is_all_oto, one_to_one_count, total_edges

# Analyze length 5 chains
if length_5_chains:
    good_5 = 0
    for chain in length_5_chains:
        is_all_oto, oto_count, total = check_chain_quality(chain, tracker_final)
        if is_all_oto:
            good_5 += 1
    print(f"\nLength 5 with all one_to_one edges: {good_5}")

# Analyze length 6 chains  
if length_6_chains:
    good_6 = 0
    for chain in length_6_chains:
        is_all_oto, oto_count, total = check_chain_quality(chain, tracker_final)
        if is_all_oto:
            good_6 += 1
    print(f"Length 6 with all one_to_one edges: {good_6}")

print("\n" + "="*70)
print("SUMMARY:")
print("="*70)
print(f"In our extracted high-quality chains (length 7 only): 0 length 5/6 chains")
print(f"In the full graph (including partial chains):")
print(f"  - Length 5 chains: {len(length_5_chains)} total, {good_5 if length_5_chains else 0} with all one_to_one edges")
print(f"  - Length 6 chains: {len(length_6_chains)} total, {good_6 if length_6_chains else 0} with all one_to_one edges")

In [ ]:
# Summary: All 43 Tracked Chains with Coordinates
print(f"\n{'='*70}")
print(f"SUMMARY: All {len(all_extracted_chains)} Tracked Chains")
print(f"{'='*70}\n")

for idx, chain_data in enumerate(all_extracted_chains, 1):
    # Extract chain (list of nodes)
    if 'edges' in chain_data:
        # Extracted backbone
        chain = [chain_data['edges'][0]['node1']] + [e['node2'] for e in chain_data['edges']]
        avg_sim = chain_data['avg_similarity']
        oto_count = chain_data['one_to_one_count']
        
        if oto_count == 6:
            quality = "Excellent ★★★"
        elif oto_count >= 4:
            quality = "Good ★★"
        else:
            quality = "Fair ★"
    else:
        # Clean chain
        chain = chain_data
        quality = "Excellent ★★★ (Clean)"
        avg_sim = None
    
    # Get coordinates for first and last crown
    om_id_first, crown_idx_first = chain[0]
    om_id_last, crown_idx_last = chain[-1]
    
    first_crown = tracker_final.crowns_gdfs[om_id_first].iloc[crown_idx_first]
    last_crown = tracker_final.crowns_gdfs[om_id_last].iloc[crown_idx_last]
    
    centroid_first = first_crown.geometry.centroid
    centroid_last = last_crown.geometry.centroid
    
    coords_start = f"({centroid_first.x:.1f}, {centroid_first.y:.1f})"
    coords_end = f"({centroid_last.x:.1f}, {centroid_last.y:.1f})"
    
    # Calculate movement
    dx = centroid_last.x - centroid_first.x
    dy = centroid_last.y - centroid_first.y
    distance = (dx**2 + dy**2)**0.5
    
    sim_str = f"Sim={avg_sim:.3f}" if avg_sim is not None else "No branching"
    
    print(f"Chain {idx:2d}: {coords_start} → {coords_end} | Δ={distance:5.1f}m | {quality} | {sim_str}")

print(f"\n{'='*70}")
print(f"✓ All visualizations saved to: output/tracked_chains_visualization/")
print(f"{'='*70}")

In [ ]:
# Display sample visualizations (first 3 chains)
from IPython.display import Image, display
import os

print("Sample visualizations (first 3 chains):\n")

for i in range(1, min(4, len(all_extracted_chains) + 1)):
    img_path = f"../../output/tracked_chains_visualization/chain_{i:02d}.png"
    if os.path.exists(img_path):
        print(f"\nChain {i}:")
        display(Image(filename=img_path, width=1200))
    else:
        print(f"Chain {i}: Image not found at {img_path}")

---

## ✅ Multi-Threshold Crown Tracking: Complete Results

### Summary
Successfully tracked **43 high-quality tree crowns** across all 7 orthomosaic time points (OM1-OM7).

### Key Achievements

1. **Image Extraction Strategy**
   - Images extracted from **original crown positions** BEFORE spatial alignment
   - Alignment shifts applied AFTER image extraction
   - This ensures tree images correspond to actual orthomosaic locations
   - Each crown has its RGB image patch for visualization

2. **Chain Quality Distribution**
   - **Clean chains (no branching):** 2 chains
   - **Extracted backbones:** 41 chains
     - Pure one_to_one paths (100% best matches): 32 chains
     - Mixed-edge paths (≥77% one_to_one): 9 chains
   
3. **Quality Metrics**
   - Pure one_to_one avg similarity: **0.889**
   - Mixed-edge avg similarity: **0.818**
   - Mixed-edge paths maintain high quality with avg 77.8% one_to_one edges

4. **Visualization Outputs**
   - **Location:** `output/tracked_chains_visualization/`
   - **Format:** PNG images (chain_01.png to chain_43.png)
   - **Content:** Each visualization shows:
     - Row 1: Crown polygons with coordinates and edge information
     - Row 2: Extracted RGB tree images for all 7 time points
   - **File size:** ~2-5 MB per visualization
   - **Total size:** ~185 MB for all 43 chains

### Next Steps
These 43 tracked chains with their extracted tree images are ready for:
- Phenology signal extraction
- Temporal change analysis
- Consensus crown generation
- Time-series visualization

In [ ]:
# Create a compact summary table of all chains
import pandas as pd

chain_summary = []

for idx, chain_data in enumerate(all_extracted_chains, 1):
    # Extract chain info
    if 'edges' in chain_data:
        chain = [chain_data['edges'][0]['node1']] + [e['node2'] for e in chain_data['edges']]
        avg_sim = chain_data['avg_similarity']
        oto_count = chain_data['one_to_one_count']
        quality = "Pure" if oto_count == 6 else "Mixed"
    else:
        chain = chain_data
        avg_sim = 1.0
        oto_count = 6
        quality = "Clean"
    
    # Get coordinates
    om_id_first, crown_idx_first = chain[0]
    first_crown = tracker_final.crowns_gdfs[om_id_first].iloc[crown_idx_first]
    centroid = first_crown.geometry.centroid
    
    chain_summary.append({
        'Chain': idx,
        'Quality': quality,
        'Similarity': f"{avg_sim:.3f}",
        'One-to-One': f"{oto_count}/6",
        'Start_X': f"{centroid.x:.1f}",
        'Start_Y': f"{centroid.y:.1f}",
        'Area_m2': f"{first_crown.geometry.area:.1f}",
        'Visualization': f"chain_{idx:02d}.png"
    })

df_summary = pd.DataFrame(chain_summary)
print(f"\n{'='*80}")
print(f"Complete Summary of All 43 Tracked Crown Chains")
print(f"{'='*80}\n")
print(df_summary.to_string(index=False))
print(f"\n{'='*80}")
print(f"Files saved to: output/tracked_chains_visualization/")
print(f"{'='*80}")

## Consensus Crown Generation

Generate consensus crown polygons for all tracked chains using the medoid strategy.

In [ ]:
from shapely.ops import unary_union
from shapely.affinity import translate
from functools import reduce

def _chain_polygons_aligned(chain, tracker):
    """Extract all polygons from a chain."""
    polys = []
    for om_id, crown_idx in chain:
        poly = tracker.crowns_gdfs[om_id].iloc[crown_idx].geometry
        if poly is not None and not poly.is_empty:
            polys.append(poly.buffer(0))
    return polys

def _largest_polygon(geom):
    """Extract largest polygon from geometry collection."""
    if geom is None or geom.is_empty:
        return geom
    if geom.geom_type == "Polygon":
        return geom
    if geom.geom_type in ("MultiPolygon", "GeometryCollection"):
        polys = [g for g in geom.geoms if g.geom_type == "Polygon"]
        return max(polys, key=lambda g: g.area) if polys else geom
    return geom

def consensus_medoid(chain, tracker, w_centroid=0.5, w_iou=0.4, w_area=0.1):
    """
    Compute consensus crown using medoid strategy.
    Selects the crown polygon that minimizes dissimilarity to all other crowns in the chain.
    
    Args:
        chain: List of (om_id, crown_idx) tuples
        tracker: TreeTrackingGraph instance
        w_centroid: Weight for centroid distance
        w_iou: Weight for IoU similarity
        w_area: Weight for area ratio
    
    Returns:
        Shapely Polygon representing the consensus crown
    """
    polys = _chain_polygons_aligned(chain, tracker)
    if not polys:
        return None
    
    centroids = [p.centroid for p in polys]
    areas = [p.area for p in polys]
    max_dist = max((c1.distance(c2) for c1 in centroids for c2 in centroids), default=1.0) or 1.0
    
    best_idx = 0
    best_score = float("inf")
    
    for i, p in enumerate(polys):
        score = 0.0
        for j, q in enumerate(polys):
            if i == j:
                continue
            dist = centroids[i].distance(centroids[j]) / max_dist
            iou = tracker._compute_iou(p, q)
            area_ratio = min(areas[i], areas[j]) / max(areas[i], areas[j]) if max(areas[i], areas[j]) > 0 else 0.0
            score += (w_centroid * dist) + (w_iou * (1.0 - iou)) + (w_area * (1.0 - area_ratio))
        
        if score < best_score:
            best_score = score
            best_idx = i
    
    return polys[best_idx]

def extract_patch_for_polygon(om_id, polygon_aligned, tracker):
    """
    Extract an image patch for a consensus polygon from the original orthomosaic.
    
    Args:
        om_id: Orthomosaic ID
        polygon_aligned: Aligned polygon geometry
        tracker: TreeTrackingGraph instance
    
    Returns:
        numpy array with RGB image patch or None
    """
    if polygon_aligned is None or polygon_aligned.is_empty:
        return None
    
    # Get alignment shift for this OM (reverse translation to get original position)
    dx, dy = tracker.alignment_shifts.get(om_id, (0.0, 0.0))
    polygon_original = translate(polygon_aligned, xoff=-dx, yoff=-dy)
    
    # Find orthomosaic file
    ortho_file = None
    for oid, (crown_file, of) in zip(tracker.om_ids, tracker.file_pairs):
        if oid == om_id:
            ortho_file = of
            break
    
    if not ortho_file or not os.path.exists(ortho_file):
        return None
    
    try:
        with rasterio.open(ortho_file) as src:
            from rasterio.mask import mask
            from shapely.geometry import mapping
            out_image, _ = mask(src, [mapping(polygon_original)], crop=True)
            return np.moveaxis(out_image, 0, -1)
    except Exception:
        return None

print("Consensus crown functions loaded")

In [ ]:
# Generate consensus crowns for all 35 tracked chains
print(f"\n{'='*70}")
print(f"Generating Consensus Crowns for All {len(all_extracted_chains)} Chains")
print(f"{'='*70}\n")

consensus_geoms = []
chain_ids = []
chain_lengths = []
chain_qualities = []
avg_similarities = []

for idx, chain_data in enumerate(all_extracted_chains, 1):
    # Extract chain (list of nodes)
    if 'edges' in chain_data:
        # Extracted backbone
        chain = [chain_data['edges'][0]['node1']] + [e['node2'] for e in chain_data['edges']]
        avg_sim = chain_data['avg_similarity']
        oto_count = chain_data['one_to_one_count']
        quality = "Pure" if oto_count == 6 else "Mixed"
    else:
        # Clean chain
        chain = chain_data
        avg_sim = 1.0
        oto_count = 6
        quality = "Clean"
    
    # Generate consensus polygon using medoid strategy
    consensus_poly = consensus_medoid(chain, tracker_final)
    
    if consensus_poly is not None and not consensus_poly.is_empty:
        consensus_geoms.append(consensus_poly)
        chain_ids.append(idx)
        chain_lengths.append(len(chain))
        chain_qualities.append(quality)
        avg_similarities.append(avg_sim)
        
        if idx <= 3:
            print(f"Chain {idx}: Consensus area = {consensus_poly.area:.1f} m², quality = {quality}")

# Create GeoDataFrame
base_crs = tracker_final.crowns_gdfs[tracker_final.om_ids[0]].crs

consensus_crowns_gdf = gpd.GeoDataFrame({
    'chain_id': chain_ids,
    'chain_length': chain_lengths,
    'quality': chain_qualities,
    'avg_similarity': avg_similarities,
    'geometry': consensus_geoms
}, crs=base_crs)

print(f"\n✓ Generated {len(consensus_crowns_gdf)} consensus crowns")
print(f"  - Clean chains: {(consensus_crowns_gdf['quality'] == 'Clean').sum()}")
print(f"  - Pure one_to_one: {(consensus_crowns_gdf['quality'] == 'Pure').sum()}")
print(f"  - Mixed-edge: {(consensus_crowns_gdf['quality'] == 'Mixed').sum()}")
print(f"\nConsensus crown statistics:")
print(consensus_crowns_gdf[['chain_id', 'quality', 'avg_similarity']].head(10))

In [ ]:
def visualize_consensus_chain(chain, consensus_poly, tracker, chain_id, quality, avg_sim, save_path=None):
    """
    Visualize a consensus crown overlaid on all original crowns in the chain,
    with extracted RGB patches from the consensus polygon.
    
    Args:
        chain: List of (om_id, crown_idx) tuples
        consensus_poly: Consensus polygon geometry
        tracker: TreeTrackingGraph instance
        chain_id: Chain ID number
        quality: Chain quality (Clean/Pure/Mixed)
        avg_sim: Average similarity score
        save_path: Path to save visualization
    """
    if consensus_poly is None:
        print(f"Chain {chain_id}: No consensus polygon available")
        return
    
    chain_length = len(chain)
    fig = plt.figure(figsize=(5 * chain_length, 10))
    
    # Calculate consensus crown centroid
    consensus_centroid = consensus_poly.centroid
    
    for idx, (om_id, crown_idx) in enumerate(chain):
        gdf = tracker.crowns_gdfs[om_id]
        crown = gdf.iloc[crown_idx]
        
        # Top row: Polygon overlay
        ax_poly = plt.subplot(2, chain_length, idx + 1)
        
        # Plot all crowns in light gray
        gdf.plot(ax=ax_poly, color='lightgray', edgecolor='gray', alpha=0.3, linewidth=0.5)
        
        # Plot consensus polygon in red
        gpd.GeoSeries([consensus_poly]).plot(
            ax=ax_poly,
            facecolor='red',
            edgecolor='darkred',
            linewidth=2.5,
            alpha=0.4
        )
        
        # Plot original crown in black
        gpd.GeoSeries([crown.geometry]).plot(
            ax=ax_poly,
            facecolor='none',
            edgecolor='black',
            linewidth=2.0,
            alpha=0.9
        )
        
        # Mark consensus centroid
        ax_poly.plot(consensus_centroid.x, consensus_centroid.y, 'o', 
                    color='yellow', markersize=10,
                    markeredgecolor='black', markeredgewidth=2)
        
        # Calculate IoU between consensus and original crown
        iou = tracker._compute_iou(consensus_poly, crown.geometry)
        
        ax_poly.set_aspect('equal')
        ax_poly.set_title(f"OM{om_id}\nIoU={iou:.3f}", fontsize=11, fontweight='bold')
        ax_poly.grid(True, alpha=0.3)
        
        # Bottom row: RGB image extracted using consensus polygon
        ax_img = plt.subplot(2, chain_length, chain_length + idx + 1)
        patch = extract_patch_for_polygon(om_id, consensus_poly, tracker)
        
        if patch is not None and patch.size > 0:
            # Clip values to valid range [0, 255] for uint8 display
            patch_display = np.clip(patch[:,:,:3], 0, 255).astype(np.uint8)
            ax_img.imshow(patch_display)
            ax_img.set_title(f"Consensus Patch\n{patch_display.shape[0]}×{patch_display.shape[1]} px", 
                           fontsize=10)
        else:
            ax_img.text(0.5, 0.5, 'Image not available', 
                       ha='center', va='center', fontsize=10, color='red')
            ax_img.set_title("No Image", fontsize=9)
        ax_img.axis('off')
    
    # Add overall title
    fig.suptitle(f"Consensus Crown Chain {chain_id} | Quality: {quality} | Avg Similarity: {avg_sim:.3f}",
                fontsize=16, fontweight='bold', y=0.98)
    
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close()
    else:
        plt.show()

print("Consensus visualization function loaded")

In [ ]:
# Generate and save consensus crown visualizations for all chains
output_dir_consensus = "../../output/consensus_crowns_visualization"
os.makedirs(output_dir_consensus, exist_ok=True)

print(f"\n{'='*70}")
print(f"Generating Consensus Crown Visualizations for All {len(all_extracted_chains)} Chains")
print(f"{'='*70}\n")

for idx, chain_data in enumerate(all_extracted_chains, 1):
    # Extract chain info
    if 'edges' in chain_data:
        chain = [chain_data['edges'][0]['node1']] + [e['node2'] for e in chain_data['edges']]
        avg_sim = chain_data['avg_similarity']
        oto_count = chain_data['one_to_one_count']
        quality = "Pure" if oto_count == 6 else "Mixed"
    else:
        chain = chain_data
        avg_sim = 1.0
        quality = "Clean"
    
    # Get consensus polygon for this chain
    consensus_poly = consensus_medoid(chain, tracker_final)
    
    # Create visualization
    save_path = os.path.join(output_dir_consensus, f"consensus_chain_{idx:02d}.png")
    visualize_consensus_chain(
        chain,
        consensus_poly,
        tracker_final,
        chain_id=idx,
        quality=quality,
        avg_sim=avg_sim,
        save_path=save_path
    )
    
    if idx % 10 == 0:
        print(f"  ✓ Generated {idx}/{len(all_extracted_chains)} visualizations")

print(f"\n{'='*70}")
print(f"✓ All {len(all_extracted_chains)} consensus crown visualizations saved")
print(f"  Location: {output_dir_consensus}/")
print(f"  Files: consensus_chain_01.png to consensus_chain_{len(all_extracted_chains):02d}.png")
print(f"{'='*70}")